# HYGD OD/OC proxy generation

**Public role:** Core frozen-proxy inference and quality-control notebook.

**Relation to the manuscript:** Generates the HYGD OD/OC proxy and strict-valid QC used by AQPR.

**Execution status of this curated copy:** Stored outputs and execution counters were removed to prevent disclosure of local paths, embedded dataset images, stale errors, and large widget state. The code was not re-executed after this mechanical curation. Reference outputs are distributed separately under `results/`.

**Path configuration:** Set `AQPR_PROJECT_ROOT` to the working directory containing `project_data/`, `models/`, and `results/`. Set `AQPR_DATA_ROOT` to the external dataset root. The default values target a Google Drive/Colab workflow and can be changed without editing the notebook logic.

**Original source notebook:** `06_apply_final_od_oc_segmentor_to_hygd_anatomical_proxy.ipynb`


In [ ]:
import os
# Cell 1: Load project paths, final segmentor decision, and HYGD metadata

from pathlib import Path
import json
import time
import shutil

import cv2
import numpy as np
import pandas as pd

import torch

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

JOURNAL_ROOT = Path(os.environ.get("AQPR_PROJECT_ROOT", "/content/drive/MyDrive/AQPR_Glaucoma"))
CONFIG_PATH = JOURNAL_ROOT / "configs" / "project_config.json"
PATH_REGISTRY_PATH = JOURNAL_ROOT / "configs" / "path_registry.csv"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Missing project config: {CONFIG_PATH}")

if not PATH_REGISTRY_PATH.exists():
    raise FileNotFoundError(f"Missing path registry: {PATH_REGISTRY_PATH}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)

path_registry = pd.read_csv(PATH_REGISTRY_PATH)
PATHS = {
    row["path_key"]: Path(row["path"])
    for _, row in path_registry.iterrows()
}

RESULTS_06_DIR = JOURNAL_ROOT / "results" / "06_hygd_final_od_oc_proxy"
RESULTS_06_DIR.mkdir(parents=True, exist_ok=True)

HYGD_PROXY_ROOT = JOURNAL_ROOT / "project_data" / "hygd_final_od_oc_proxy"
HYGD_MASK_DIR = HYGD_PROXY_ROOT / "masks_512"
HYGD_OVERLAY_DIR = HYGD_PROXY_ROOT / "overlays_512"
HYGD_PROB_DIR = HYGD_PROXY_ROOT / "probability_maps_512"

for directory in [HYGD_PROXY_ROOT, HYGD_MASK_DIR, HYGD_OVERLAY_DIR, HYGD_PROB_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

FINAL_DECISION_JSON = (
    JOURNAL_ROOT /
    "results" /
    "05_od_oc_segmentor_training" /
    "notebook05_final_segmentor_decision.json"
)

if not FINAL_DECISION_JSON.exists():
    raise FileNotFoundError(f"Missing final segmentor decision: {FINAL_DECISION_JSON}")

with open(FINAL_DECISION_JSON, "r", encoding="utf-8") as f:
    FINAL_SEGMENTOR_DECISION = json.load(f)

FINAL_MODEL_NAME = FINAL_SEGMENTOR_DECISION["final_segmentor_model"]
FINAL_CHECKPOINT_PATH = Path(FINAL_SEGMENTOR_DECISION["final_checkpoint"])

if FINAL_MODEL_NAME != "ResidualUNetLite":
    raise RuntimeError(f"Unexpected final model: {FINAL_MODEL_NAME}")

if not FINAL_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Missing final checkpoint: {FINAL_CHECKPOINT_PATH}")

METADATA_CLEAN_CSV = JOURNAL_ROOT / "project_data" / "metadata" / "metadata_clean.csv"

if not METADATA_CLEAN_CSV.exists():
    raise FileNotFoundError(f"Missing HYGD clean metadata: {METADATA_CLEAN_CSV}")

hygd_metadata = pd.read_csv(METADATA_CLEAN_CSV)

candidate_image_path_columns = [
    "image_path",
    "Image Path",
    "path",
    "full_image_path",
]

image_path_col = None

for col in candidate_image_path_columns:
    if col in hygd_metadata.columns:
        image_path_col = col
        break

if image_path_col is None:
    raise ValueError(
        f"No image path column found. Available columns: {hygd_metadata.columns.tolist()}"
    )

hygd_metadata["image_path_for_inference"] = hygd_metadata[image_path_col].astype(str)
hygd_metadata["image_exists"] = hygd_metadata["image_path_for_inference"].apply(lambda x: Path(x).exists())

missing_images = int((~hygd_metadata["image_exists"]).sum())

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()

INPUT_SIZE = 512
NUM_CLASSES = 3
CLASS_NAMES = {
    0: "background",
    1: "optic_disc",
    2: "optic_cup",
}

setup_summary = {
    "notebook": "06_apply_final_od_oc_segmentor_to_hygd_anatomical_proxy",
    "device": str(DEVICE),
    "cuda_available": bool(torch.cuda.is_available()),
    "use_amp": bool(USE_AMP),
    "final_model_name": FINAL_MODEL_NAME,
    "final_checkpoint": str(FINAL_CHECKPOINT_PATH),
    "hygd_metadata_rows": int(len(hygd_metadata)),
    "image_path_column": image_path_col,
    "missing_images": missing_images,
    "mask_output_dir": str(HYGD_MASK_DIR),
    "overlay_output_dir": str(HYGD_OVERLAY_DIR),
    "probability_output_dir": str(HYGD_PROB_DIR),
}

setup_summary_json = RESULTS_06_DIR / "notebook06_setup_summary.json"

with open(setup_summary_json, "w", encoding="utf-8") as f:
    json.dump(setup_summary, f, indent=2)

print(f"Device: {DEVICE}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"AMP enabled: {USE_AMP}")
print(f"Final model: {FINAL_MODEL_NAME}")
print(f"HYGD metadata rows: {len(hygd_metadata)}")
print(f"Missing images: {missing_images}")
print(f"Setup summary: {setup_summary_json}")

if len(hygd_metadata) != 747:
    raise RuntimeError(f"Expected 747 HYGD images, found {len(hygd_metadata)}")

if missing_images > 0:
    raise RuntimeError(f"Missing HYGD image files: {missing_images}")

print("Status: PASS")

In [ ]:
# Cell 2: Define final ResidualUNetLite model and run one-image inference smoke test

import torch.nn as nn
import torch.nn.functional as F
from contextlib import nullcontext


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )

        self.skip = (
            nn.Identity()
            if in_channels == out_channels
            else nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        )

    def forward(self, x):
        return F.relu(self.conv(x) + self.skip(x), inplace=True)


class ResidualUNetLite(nn.Module):
    def __init__(self, in_channels=3, num_classes=3, base_channels=24):
        super().__init__()

        self.enc1 = ResidualBlock(in_channels, base_channels)
        self.enc2 = ResidualBlock(base_channels, base_channels * 2)
        self.enc3 = ResidualBlock(base_channels * 2, base_channels * 4)
        self.enc4 = ResidualBlock(base_channels * 4, base_channels * 8)

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = ResidualBlock(base_channels * 8, base_channels * 16)

        self.up4 = nn.ConvTranspose2d(base_channels * 16, base_channels * 8, 2, 2)
        self.dec4 = ResidualBlock(base_channels * 16, base_channels * 8)

        self.up3 = nn.ConvTranspose2d(base_channels * 8, base_channels * 4, 2, 2)
        self.dec3 = ResidualBlock(base_channels * 8, base_channels * 4)

        self.up2 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, 2, 2)
        self.dec2 = ResidualBlock(base_channels * 4, base_channels * 2)

        self.up1 = nn.ConvTranspose2d(base_channels * 2, base_channels, 2, 2)
        self.dec1 = ResidualBlock(base_channels * 2, base_channels)

        self.head = nn.Conv2d(base_channels, num_classes, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.head(d1)


def preprocess_image_for_segmentor(image_bgr, input_size=512):
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    image_rgb_512 = cv2.resize(image_rgb, (input_size, input_size), interpolation=cv2.INTER_AREA)

    image_tensor = image_rgb_512.astype(np.float32) / 255.0
    image_tensor = np.transpose(image_tensor, (2, 0, 1))
    image_tensor = torch.from_numpy(np.ascontiguousarray(image_tensor)).float().unsqueeze(0)

    return image_tensor, image_rgb_512


def inference_context():
    if USE_AMP and DEVICE.type == "cuda":
        return torch.amp.autocast(device_type="cuda", enabled=True)
    return nullcontext()


final_segmentor = ResidualUNetLite(
    in_channels=3,
    num_classes=NUM_CLASSES,
    base_channels=24,
).to(DEVICE)

checkpoint = torch.load(
    FINAL_CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False,
)

final_segmentor.load_state_dict(checkpoint["model_state_dict"])
final_segmentor.eval()

n_params = sum(p.numel() for p in final_segmentor.parameters())
n_trainable = sum(p.numel() for p in final_segmentor.parameters() if p.requires_grad)

sample_row = hygd_metadata.iloc[0]
sample_image_path = sample_row["image_path_for_inference"]
sample_id = (
    sample_row["image_id"]
    if "image_id" in hygd_metadata.columns
    else Path(sample_image_path).stem
)

sample_image_bgr = cv2.imread(sample_image_path, cv2.IMREAD_COLOR)

if sample_image_bgr is None:
    raise RuntimeError(f"Unreadable HYGD sample image: {sample_image_path}")

image_tensor, image_rgb_512 = preprocess_image_for_segmentor(
    sample_image_bgr,
    input_size=INPUT_SIZE,
)

image_tensor = image_tensor.to(DEVICE)

with torch.no_grad():
    with inference_context():
        logits = final_segmentor(image_tensor)
        probabilities = torch.softmax(logits, dim=1)
        pred_mask = torch.argmax(probabilities, dim=1)

pred_mask_np = pred_mask.squeeze(0).detach().cpu().numpy().astype(np.uint8)
probabilities_np = probabilities.squeeze(0).detach().cpu().numpy()

mask_values = sorted(np.unique(pred_mask_np).astype(int).tolist())

od_pixels = int((pred_mask_np == 1).sum())
oc_pixels = int((pred_mask_np == 2).sum())

model_smoke_summary = {
    "model_name": FINAL_MODEL_NAME,
    "checkpoint_path": str(FINAL_CHECKPOINT_PATH),
    "device": str(DEVICE),
    "n_parameters": int(n_params),
    "n_trainable_parameters": int(n_trainable),
    "sample_id": str(sample_id),
    "sample_image_path": str(sample_image_path),
    "input_tensor_shape": list(image_tensor.shape),
    "logits_shape": list(logits.shape),
    "probabilities_shape": list(probabilities.shape),
    "pred_mask_shape": list(pred_mask_np.shape),
    "pred_mask_values": mask_values,
    "od_pixels_512": od_pixels,
    "oc_pixels_512": oc_pixels,
}

model_smoke_json = RESULTS_06_DIR / "notebook06_model_loading_smoke_test.json"

with open(model_smoke_json, "w", encoding="utf-8") as f:
    json.dump(model_smoke_summary, f, indent=2)

print(f"Model: {FINAL_MODEL_NAME}")
print(f"Parameters: {n_params}")
print(f"Trainable parameters: {n_trainable}")
print(f"Sample ID: {sample_id}")
print(f"Input tensor shape: {tuple(image_tensor.shape)}")
print(f"Logits shape: {tuple(logits.shape)}")
print(f"Pred mask shape: {pred_mask_np.shape}")
print(f"Pred mask values: {mask_values}")
print(f"OD pixels 512: {od_pixels}")
print(f"OC pixels 512: {oc_pixels}")
print(f"Smoke test: {model_smoke_json}")

if tuple(logits.shape) != (1, NUM_CLASSES, INPUT_SIZE, INPUT_SIZE):
    raise RuntimeError(f"Unexpected logits shape: {tuple(logits.shape)}")

if od_pixels == 0:
    raise RuntimeError("Smoke test produced empty OD mask.")

if oc_pixels == 0:
    raise RuntimeError("Smoke test produced empty OC mask.")

print("Status: PASS")

In [ ]:
# Cell 3: Define anatomical postprocessing and run one-sample proxy smoke test

def largest_connected_component(binary_mask):
    binary_mask = (binary_mask > 0).astype(np.uint8)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary_mask, connectivity=8)

    if num_labels <= 1:
        return np.zeros_like(binary_mask, dtype=np.uint8), 0, 0

    component_areas = stats[1:, cv2.CC_STAT_AREA]
    largest_label = 1 + int(np.argmax(component_areas))
    largest_area = int(component_areas.max())

    largest = (labels == largest_label).astype(np.uint8)

    return largest, int(num_labels - 1), largest_area


def compute_mask_centroid(binary_mask):
    ys, xs = np.where(binary_mask > 0)

    if len(xs) == 0:
        return np.nan, np.nan

    return float(xs.mean()), float(ys.mean())


def compute_mask_bbox(binary_mask):
    ys, xs = np.where(binary_mask > 0)

    if len(xs) == 0:
        return {
            "x_min": np.nan,
            "y_min": np.nan,
            "x_max": np.nan,
            "y_max": np.nan,
            "width": np.nan,
            "height": np.nan,
        }

    x_min = int(xs.min())
    x_max = int(xs.max())
    y_min = int(ys.min())
    y_max = int(ys.max())

    return {
        "x_min": x_min,
        "y_min": y_min,
        "x_max": x_max,
        "y_max": y_max,
        "width": int(x_max - x_min + 1),
        "height": int(y_max - y_min + 1),
    }


def postprocess_od_oc_mask(raw_mask):
    raw_disc_region = (raw_mask > 0).astype(np.uint8)
    raw_oc = (raw_mask == 2).astype(np.uint8)

    disc_region, disc_components, disc_largest_area = largest_connected_component(raw_disc_region)
    oc_mask, oc_components, oc_largest_area = largest_connected_component(raw_oc)

    oc_outside_disc_before = int((oc_mask & (1 - disc_region)).sum())

    if disc_region.sum() > 0 and oc_mask.sum() > 0:
        clipped_oc = (oc_mask & disc_region).astype(np.uint8)
        clipped_oc, clipped_oc_components, clipped_oc_largest_area = largest_connected_component(clipped_oc)

        if clipped_oc.sum() > 0:
            oc_mask = clipped_oc
            oc_clipped_to_disc = True
        else:
            oc_clipped_to_disc = False
    else:
        clipped_oc_components = 0
        clipped_oc_largest_area = 0
        oc_clipped_to_disc = False

    disc_rim = (disc_region & (1 - oc_mask)).astype(np.uint8)

    clean_mask = np.zeros_like(raw_mask, dtype=np.uint8)
    clean_mask[disc_rim == 1] = 1
    clean_mask[oc_mask == 1] = 2

    return {
        "clean_mask": clean_mask,
        "disc_region_mask": disc_region,
        "disc_rim_mask": disc_rim,
        "oc_mask": oc_mask,
        "raw_disc_components": disc_components,
        "raw_oc_components": oc_components,
        "disc_largest_area": disc_largest_area,
        "oc_largest_area": oc_largest_area,
        "oc_outside_disc_before": oc_outside_disc_before,
        "oc_clipped_to_disc": oc_clipped_to_disc,
        "clipped_oc_components": clipped_oc_components,
        "clipped_oc_largest_area": clipped_oc_largest_area,
    }


def compute_anatomical_proxy_metrics(sample_id, clean_mask, probabilities):
    od_mask = (clean_mask > 0).astype(np.uint8)
    oc_mask = (clean_mask == 2).astype(np.uint8)
    rim_mask = (clean_mask == 1).astype(np.uint8)

    od_area = int(od_mask.sum())
    oc_area = int(oc_mask.sum())
    rim_area = int(rim_mask.sum())

    od_area_ratio = float(od_area / (INPUT_SIZE * INPUT_SIZE))
    oc_area_ratio = float(oc_area / (INPUT_SIZE * INPUT_SIZE))
    rim_area_ratio = float(rim_area / (INPUT_SIZE * INPUT_SIZE))

    cup_to_disc_area_ratio = float(oc_area / od_area) if od_area > 0 else np.nan
    area_cdr_proxy = float(np.sqrt(cup_to_disc_area_ratio)) if od_area > 0 and oc_area > 0 else np.nan

    od_cx, od_cy = compute_mask_centroid(od_mask)
    oc_cx, oc_cy = compute_mask_centroid(oc_mask)

    if np.isfinite(od_cx) and np.isfinite(oc_cx):
        centroid_distance_px = float(np.sqrt((od_cx - oc_cx) ** 2 + (od_cy - oc_cy) ** 2))
    else:
        centroid_distance_px = np.nan

    od_equivalent_radius = float(np.sqrt(od_area / np.pi)) if od_area > 0 else np.nan
    centroid_distance_norm_od_radius = (
        float(centroid_distance_px / od_equivalent_radius)
        if np.isfinite(centroid_distance_px) and od_equivalent_radius > 0
        else np.nan
    )

    od_bbox = compute_mask_bbox(od_mask)
    oc_bbox = compute_mask_bbox(oc_mask)

    od_prob_union = np.maximum(probabilities[1], probabilities[2])

    if od_area > 0:
        od_confidence_mean = float(od_prob_union[od_mask == 1].mean())
    else:
        od_confidence_mean = np.nan

    if oc_area > 0:
        oc_confidence_mean = float(probabilities[2][oc_mask == 1].mean())
    else:
        oc_confidence_mean = np.nan

    return {
        "sample_id": str(sample_id),
        "od_area_512": od_area,
        "oc_area_512": oc_area,
        "rim_area_512": rim_area,
        "od_area_ratio_512": od_area_ratio,
        "oc_area_ratio_512": oc_area_ratio,
        "rim_area_ratio_512": rim_area_ratio,
        "cup_to_disc_area_ratio": cup_to_disc_area_ratio,
        "area_cdr_proxy": area_cdr_proxy,
        "od_centroid_x_512": od_cx,
        "od_centroid_y_512": od_cy,
        "oc_centroid_x_512": oc_cx,
        "oc_centroid_y_512": oc_cy,
        "oc_od_centroid_distance_px": centroid_distance_px,
        "oc_od_centroid_distance_norm_od_radius": centroid_distance_norm_od_radius,
        "od_bbox_x_min": od_bbox["x_min"],
        "od_bbox_y_min": od_bbox["y_min"],
        "od_bbox_x_max": od_bbox["x_max"],
        "od_bbox_y_max": od_bbox["y_max"],
        "od_bbox_width": od_bbox["width"],
        "od_bbox_height": od_bbox["height"],
        "oc_bbox_x_min": oc_bbox["x_min"],
        "oc_bbox_y_min": oc_bbox["y_min"],
        "oc_bbox_x_max": oc_bbox["x_max"],
        "oc_bbox_y_max": oc_bbox["y_max"],
        "oc_bbox_width": oc_bbox["width"],
        "oc_bbox_height": oc_bbox["height"],
        "od_confidence_mean": od_confidence_mean,
        "oc_confidence_mean": oc_confidence_mean,
    }


def create_od_oc_overlay(image_rgb, clean_mask, alpha=0.35):
    overlay = image_rgb.copy()

    od_color = np.array([0, 255, 0], dtype=np.uint8)
    oc_color = np.array([255, 0, 0], dtype=np.uint8)

    od_mask = clean_mask > 0
    oc_mask = clean_mask == 2

    overlay[od_mask] = (
        (1.0 - alpha) * overlay[od_mask].astype(np.float32)
        + alpha * od_color.astype(np.float32)
    ).astype(np.uint8)

    overlay[oc_mask] = (
        (1.0 - alpha) * overlay[oc_mask].astype(np.float32)
        + alpha * oc_color.astype(np.float32)
    ).astype(np.uint8)

    od_contours, _ = cv2.findContours((od_mask.astype(np.uint8) * 255), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    oc_contours, _ = cv2.findContours((oc_mask.astype(np.uint8) * 255), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    overlay_bgr = cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
    cv2.drawContours(overlay_bgr, od_contours, -1, (0, 255, 0), 2)
    cv2.drawContours(overlay_bgr, oc_contours, -1, (0, 0, 255), 2)

    return overlay_bgr


postprocessed = postprocess_od_oc_mask(pred_mask_np)
clean_mask = postprocessed["clean_mask"]

sample_metrics = compute_anatomical_proxy_metrics(
    sample_id=sample_id,
    clean_mask=clean_mask,
    probabilities=probabilities_np,
)

sample_metrics.update(
    {
        "raw_disc_components": postprocessed["raw_disc_components"],
        "raw_oc_components": postprocessed["raw_oc_components"],
        "oc_outside_disc_before": postprocessed["oc_outside_disc_before"],
        "oc_clipped_to_disc": postprocessed["oc_clipped_to_disc"],
    }
)

sample_mask_path = HYGD_MASK_DIR / f"{sample_id}_od_oc_mask_512.png"
sample_overlay_path = HYGD_OVERLAY_DIR / f"{sample_id}_od_oc_overlay_512.jpg"
sample_prob_path = HYGD_PROB_DIR / f"{sample_id}_od_oc_probabilities_512.npz"

cv2.imwrite(str(sample_mask_path), clean_mask)

sample_overlay_bgr = create_od_oc_overlay(
    image_rgb=image_rgb_512,
    clean_mask=clean_mask,
    alpha=0.35,
)

cv2.imwrite(str(sample_overlay_path), sample_overlay_bgr)

np.savez_compressed(
    sample_prob_path,
    probabilities=probabilities_np.astype(np.float16),
)

sample_metrics["mask_512_path"] = str(sample_mask_path)
sample_metrics["overlay_512_path"] = str(sample_overlay_path)
sample_metrics["probability_npz_path"] = str(sample_prob_path)

sample_proxy_df = pd.DataFrame([sample_metrics])

sample_proxy_csv = RESULTS_06_DIR / "notebook06_one_sample_anatomical_proxy_smoke_test.csv"
sample_proxy_df.to_csv(sample_proxy_csv, index=False)

print(f"Sample ID: {sample_id}")
print(f"Clean mask values: {sorted(np.unique(clean_mask).astype(int).tolist())}")
print(f"OD anatomical area 512: {sample_metrics['od_area_512']}")
print(f"OC area 512: {sample_metrics['oc_area_512']}")
print(f"Rim area 512: {sample_metrics['rim_area_512']}")
print(f"Area CDR proxy: {sample_metrics['area_cdr_proxy']:.4f}")
print(f"OD confidence mean: {sample_metrics['od_confidence_mean']:.4f}")
print(f"OC confidence mean: {sample_metrics['oc_confidence_mean']:.4f}")
print(f"OC outside disc before clipping: {sample_metrics['oc_outside_disc_before']}")
print(f"Mask: {sample_mask_path}")
print(f"Overlay: {sample_overlay_path}")
print(f"Smoke test CSV: {sample_proxy_csv}")

if sample_metrics["od_area_512"] == 0:
    raise RuntimeError("Postprocessed anatomical OD mask is empty.")

if sample_metrics["oc_area_512"] == 0:
    raise RuntimeError("Postprocessed OC mask is empty.")

if sample_metrics["oc_area_512"] >= sample_metrics["od_area_512"]:
    raise RuntimeError("OC area is not smaller than anatomical OD area.")

if not sample_mask_path.exists():
    raise RuntimeError("Sample mask was not saved.")

if not sample_overlay_path.exists():
    raise RuntimeError("Sample overlay was not saved.")

print("Status: PASS")

In [ ]:
# Cell 4: Run final OD/OC segmentor over all HYGD images with integrated repair and QC

from torch.utils.data import Dataset, DataLoader

SAVE_PROBABILITY_MAPS = False
SAVE_OVERLAYS = True
FORCE_REPROCESS = False

INFERENCE_BATCH_SIZE = 8 if DEVICE.type == "cuda" else 2
INFERENCE_NUM_WORKERS = 2 if DEVICE.type == "cuda" else 0

LIVE_PROXY_CSV = RESULTS_06_DIR / "notebook06_hygd_od_oc_proxy_live.csv"
FINAL_PROXY_CSV = HYGD_PROXY_ROOT / "hygd_final_od_oc_proxy_manifest.csv"
FINAL_PROXY_WITH_QC_CSV = HYGD_PROXY_ROOT / "hygd_final_od_oc_proxy_manifest_with_qc.csv"

RESULTS_PROXY_CSV = RESULTS_06_DIR / "notebook06_hygd_final_od_oc_proxy_manifest.csv"
RESULTS_PROXY_WITH_QC_CSV = RESULTS_06_DIR / "notebook06_hygd_final_od_oc_proxy_manifest_with_qc.csv"
REPAIR_LOG_CSV = RESULTS_06_DIR / "notebook06_hygd_od_oc_proxy_repair_log.csv"
FAILED_PROXY_CASES_CSV = RESULTS_06_DIR / "notebook06_failed_anatomical_proxy_cases.csv"
LOW_CONFIDENCE_PROXY_CASES_CSV = RESULTS_06_DIR / "notebook06_low_confidence_anatomical_proxy_cases.csv"

MIN_STRICT_VALID_PROXIES = 740


def get_hygd_sample_id(row, idx):
    if "sample_id_for_inference" in row.index:
        return str(row["sample_id_for_inference"])
    if "image_id" in row.index:
        return str(row["image_id"])
    if "sample_id" in row.index:
        return str(row["sample_id"])
    return f"HYGD_{idx:04d}"


class HYGDInferenceDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True).copy()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = row["image_path_for_inference"]
        sample_id = get_hygd_sample_id(row, idx)

        image_bgr = cv2.imread(image_path, cv2.IMREAD_COLOR)

        if image_bgr is None:
            raise RuntimeError(f"Unreadable image: {image_path}")

        image_tensor, image_rgb_512 = preprocess_image_for_segmentor(
            image_bgr,
            input_size=INPUT_SIZE,
        )

        return {
            "image": image_tensor.squeeze(0),
            "image_rgb_512": torch.from_numpy(np.ascontiguousarray(image_rgb_512)).byte(),
            "sample_id": sample_id,
            "image_path": image_path,
        }


def collate_hygd_batch(batch):
    return {
        "image": torch.stack([item["image"] for item in batch], dim=0),
        "image_rgb_512": torch.stack([item["image_rgb_512"] for item in batch], dim=0),
        "sample_id": [item["sample_id"] for item in batch],
        "image_path": [item["image_path"] for item in batch],
    }


def build_probability_based_oc(disc_region, oc_probability):
    disc_region = (disc_region > 0).astype(np.uint8)
    disc_area = int(disc_region.sum())

    if disc_area < 32:
        return np.zeros_like(disc_region, dtype=np.uint8), np.nan, "failed_too_small_disc_region"

    oc_values = oc_probability[disc_region == 1]

    if len(oc_values) == 0:
        return np.zeros_like(disc_region, dtype=np.uint8), np.nan, "failed_no_oc_values"

    min_area = max(16, int(0.005 * disc_area))
    max_area = max(min_area + 1, int(0.85 * disc_area))
    target_area = min(max(int(0.18 * disc_area), min_area), max_area)

    candidates = []

    for percentile in [99.5, 99, 98, 97, 95, 92, 90, 87, 85, 82, 80, 75, 70, 65, 60, 55, 50]:
        threshold = float(np.percentile(oc_values, percentile))

        candidate = (
            (oc_probability >= threshold) &
            (disc_region == 1)
        ).astype(np.uint8)

        candidate, _, _ = largest_connected_component(candidate)
        candidate_area = int(candidate.sum())

        if min_area <= candidate_area <= max_area:
            mean_prob = float(oc_probability[candidate == 1].mean())
            area_penalty = abs(np.log((candidate_area + 1.0) / (target_area + 1.0)))
            score = area_penalty - 0.05 * mean_prob

            candidates.append(
                {
                    "mask": candidate,
                    "threshold": threshold,
                    "method": f"percentile_{percentile}",
                    "area": candidate_area,
                    "score": score,
                }
            )

    if len(candidates) > 0:
        best = sorted(candidates, key=lambda x: x["score"])[0]
        return best["mask"], best["threshold"], best["method"]

    ys, xs = np.where(disc_region == 1)

    if len(xs) < min_area:
        return np.zeros_like(disc_region, dtype=np.uint8), np.nan, "failed_small_disc_region"

    values = oc_probability[ys, xs]
    top_k = min(target_area, len(values))
    top_idx = np.argsort(values)[-top_k:]

    candidate = np.zeros_like(disc_region, dtype=np.uint8)
    candidate[ys[top_idx], xs[top_idx]] = 1
    candidate, _, _ = largest_connected_component(candidate)

    threshold = float(values[top_idx].min()) if len(top_idx) > 0 else np.nan

    return candidate, threshold, "topk_probability_fallback"


def postprocess_od_oc_mask_with_probability_repair(raw_mask, probabilities):
    base = postprocess_od_oc_mask(raw_mask)
    clean_mask = base["clean_mask"].copy()

    old_od_area = int((clean_mask > 0).sum())
    old_oc_area = int((clean_mask == 2).sum())

    repair_needed = (
        old_od_area == 0 or
        old_oc_area == 0 or
        old_oc_area >= old_od_area
    )

    repair_applied = False
    repair_reason = "none"
    repair_method = "argmax_lcc"
    repair_threshold = np.nan

    if not repair_needed:
        base["repair_applied"] = repair_applied
        base["repair_reason"] = repair_reason
        base["repair_method"] = repair_method
        base["repair_threshold"] = repair_threshold
        base["old_od_area_512"] = old_od_area
        base["old_oc_area_512"] = old_oc_area
        return base

    if old_od_area < 32:
        base["repair_applied"] = False
        base["repair_reason"] = "too_small_od_region_for_safe_repair"
        base["repair_method"] = "not_repaired"
        base["repair_threshold"] = np.nan
        base["old_od_area_512"] = old_od_area
        base["old_oc_area_512"] = old_oc_area
        return base

    if old_oc_area == 0:
        repair_reason = "empty_oc"
    elif old_oc_area >= old_od_area:
        repair_reason = "oc_area_ge_od_area"
    else:
        repair_reason = "empty_od"

    disc_region = (clean_mask > 0).astype(np.uint8)

    oc_mask, repair_threshold, repair_method = build_probability_based_oc(
        disc_region=disc_region,
        oc_probability=probabilities[2],
    )

    if disc_region.sum() > 0 and oc_mask.sum() > 0:
        oc_mask = (oc_mask & disc_region).astype(np.uint8)
        oc_mask, _, _ = largest_connected_component(oc_mask)

    if oc_mask.sum() > 0:
        disc_rim = (disc_region & (1 - oc_mask)).astype(np.uint8)

        repaired_clean_mask = np.zeros_like(raw_mask, dtype=np.uint8)
        repaired_clean_mask[disc_rim == 1] = 1
        repaired_clean_mask[oc_mask == 1] = 2

        base["clean_mask"] = repaired_clean_mask
        base["disc_region_mask"] = disc_region
        base["disc_rim_mask"] = disc_rim
        base["oc_mask"] = oc_mask
        repair_applied = True
    else:
        repair_method = f"{repair_method}|repair_failed_empty_oc"

    base["repair_applied"] = repair_applied
    base["repair_reason"] = repair_reason
    base["repair_method"] = repair_method
    base["repair_threshold"] = repair_threshold
    base["old_od_area_512"] = old_od_area
    base["old_oc_area_512"] = old_oc_area

    return base


def assign_proxy_qc_reason(row):
    reasons = []

    if not row["mask_exists"]:
        reasons.append("missing_mask_file")

    if row["od_area_512"] <= 0:
        reasons.append("empty_od")

    if row["oc_area_512"] <= 0:
        reasons.append("empty_oc")

    if row["od_area_512"] > 0 and row["oc_area_512"] >= row["od_area_512"]:
        reasons.append("oc_area_ge_od_area")

    if not row["valid_cdr_range"]:
        reasons.append("invalid_cdr_range")

    if len(reasons) == 0:
        return "valid"

    return "|".join(reasons)


if FORCE_REPROCESS:
    for path in [
        LIVE_PROXY_CSV,
        FINAL_PROXY_CSV,
        FINAL_PROXY_WITH_QC_CSV,
        RESULTS_PROXY_CSV,
        RESULTS_PROXY_WITH_QC_CSV,
        REPAIR_LOG_CSV,
        FAILED_PROXY_CASES_CSV,
        LOW_CONFIDENCE_PROXY_CASES_CSV,
    ]:
        if path.exists():
            path.unlink()


hygd_metadata_for_inference = hygd_metadata.copy().reset_index(drop=True)

hygd_metadata_for_inference["sample_id_for_inference"] = [
    get_hygd_sample_id(row, idx)
    for idx, row in hygd_metadata_for_inference.iterrows()
]

processed_sample_ids = set()

if LIVE_PROXY_CSV.exists():
    live_existing = pd.read_csv(LIVE_PROXY_CSV)

    if "sample_id" in live_existing.columns:
        processed_sample_ids = set(live_existing["sample_id"].astype(str).tolist())
else:
    live_existing = pd.DataFrame()

pending_metadata = hygd_metadata_for_inference[
    ~hygd_metadata_for_inference["sample_id_for_inference"].astype(str).isin(processed_sample_ids)
].copy().reset_index(drop=True)

inference_dataset = HYGDInferenceDataset(pending_metadata)

inference_loader = DataLoader(
    inference_dataset,
    batch_size=INFERENCE_BATCH_SIZE,
    shuffle=False,
    num_workers=INFERENCE_NUM_WORKERS,
    pin_memory=DEVICE.type == "cuda",
    persistent_workers=INFERENCE_NUM_WORKERS > 0,
    collate_fn=collate_hygd_batch,
)

print(f"Device: {DEVICE}")
print(f"Total HYGD images: {len(hygd_metadata_for_inference)}")
print(f"Already processed: {len(processed_sample_ids)}")
print(f"Pending images: {len(pending_metadata)}")
print(f"Batch size: {INFERENCE_BATCH_SIZE}")
print(f"Save overlays: {SAVE_OVERLAYS}")
print(f"Save probability maps: {SAVE_PROBABILITY_MAPS}")

batch_rows = []
repair_log_rows = []

final_segmentor.eval()
start_time = time.time()

with torch.no_grad():
    for batch_idx, batch in enumerate(inference_loader):
        images = batch["image"].to(DEVICE, non_blocking=True)

        with inference_context():
            logits = final_segmentor(images)
            probabilities = torch.softmax(logits, dim=1)
            pred_masks = torch.argmax(probabilities, dim=1)

        probabilities_np_batch = probabilities.detach().cpu().numpy()
        pred_masks_np_batch = pred_masks.detach().cpu().numpy().astype(np.uint8)
        image_rgb_batch = batch["image_rgb_512"].numpy()

        for i, sample_id in enumerate(batch["sample_id"]):
            raw_mask = pred_masks_np_batch[i]
            probabilities_np = probabilities_np_batch[i]
            image_rgb_512_i = image_rgb_batch[i]

            postprocessed = postprocess_od_oc_mask_with_probability_repair(
                raw_mask=raw_mask,
                probabilities=probabilities_np,
            )

            clean_mask = postprocessed["clean_mask"]

            metrics = compute_anatomical_proxy_metrics(
                sample_id=sample_id,
                clean_mask=clean_mask,
                probabilities=probabilities_np,
            )

            mask_path = HYGD_MASK_DIR / f"{sample_id}_od_oc_mask_512.png"
            overlay_path = HYGD_OVERLAY_DIR / f"{sample_id}_od_oc_overlay_512.jpg"
            prob_path = HYGD_PROB_DIR / f"{sample_id}_od_oc_probabilities_512.npz"

            cv2.imwrite(str(mask_path), clean_mask)

            if SAVE_OVERLAYS:
                overlay_bgr = create_od_oc_overlay(
                    image_rgb=image_rgb_512_i,
                    clean_mask=clean_mask,
                    alpha=0.35,
                )
                cv2.imwrite(str(overlay_path), overlay_bgr)
                metrics["overlay_512_path"] = str(overlay_path)
            else:
                metrics["overlay_512_path"] = ""

            if SAVE_PROBABILITY_MAPS:
                np.savez_compressed(
                    prob_path,
                    probabilities=probabilities_np.astype(np.float16),
                )
                metrics["probability_npz_path"] = str(prob_path)
            else:
                metrics["probability_npz_path"] = ""

            metrics.update(
                {
                    "image_path": batch["image_path"][i],
                    "raw_disc_components": postprocessed["raw_disc_components"],
                    "raw_oc_components": postprocessed["raw_oc_components"],
                    "oc_outside_disc_before": postprocessed["oc_outside_disc_before"],
                    "oc_clipped_to_disc": postprocessed["oc_clipped_to_disc"],
                    "model_name": FINAL_MODEL_NAME,
                    "checkpoint_path": str(FINAL_CHECKPOINT_PATH),
                    "mask_512_path": str(mask_path),
                    "repair_applied": bool(postprocessed["repair_applied"]),
                    "repair_reason": postprocessed["repair_reason"],
                    "repair_method": postprocessed["repair_method"],
                    "repair_threshold": postprocessed["repair_threshold"],
                    "old_od_area_512": int(postprocessed["old_od_area_512"]),
                    "old_oc_area_512": int(postprocessed["old_oc_area_512"]),
                }
            )

            batch_rows.append(metrics)

            if postprocessed["repair_reason"] != "none":
                repair_log_rows.append(
                    {
                        "sample_id": sample_id,
                        "old_od_area_512": int(postprocessed["old_od_area_512"]),
                        "old_oc_area_512": int(postprocessed["old_oc_area_512"]),
                        "new_od_area_512": int(metrics["od_area_512"]),
                        "new_oc_area_512": int(metrics["oc_area_512"]),
                        "repair_applied": bool(postprocessed["repair_applied"]),
                        "repair_reason": postprocessed["repair_reason"],
                        "repair_method": postprocessed["repair_method"],
                        "repair_threshold": postprocessed["repair_threshold"],
                    }
                )

        if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == len(inference_loader):
            current_df = pd.concat(
                [
                    live_existing,
                    pd.DataFrame(batch_rows),
                ],
                axis=0,
                ignore_index=True,
            )

            current_df = current_df.drop_duplicates(
                subset=["sample_id"],
                keep="last",
            )

            current_df.to_csv(LIVE_PROXY_CSV, index=False)

            print(
                f"Processed pending batches: {batch_idx + 1}/{len(inference_loader)} | "
                f"Total saved rows: {len(current_df)}"
            )


if LIVE_PROXY_CSV.exists():
    proxy_qc = pd.read_csv(LIVE_PROXY_CSV)
else:
    proxy_qc = pd.DataFrame(batch_rows)

proxy_qc = proxy_qc.drop_duplicates(
    subset=["sample_id"],
    keep="last",
).reset_index(drop=True)

proxy_qc["sample_id"] = proxy_qc["sample_id"].astype(str)
proxy_qc["mask_exists"] = proxy_qc["mask_512_path"].apply(lambda x: Path(x).exists())

if SAVE_OVERLAYS and "overlay_512_path" in proxy_qc.columns:
    proxy_qc["overlay_exists"] = proxy_qc["overlay_512_path"].fillna("").apply(
        lambda x: Path(x).exists() if str(x).strip() else False
    )
else:
    proxy_qc["overlay_exists"] = False

proxy_qc["valid_od_nonempty"] = proxy_qc["od_area_512"] > 0
proxy_qc["valid_oc_nonempty"] = proxy_qc["oc_area_512"] > 0
proxy_qc["valid_oc_smaller_than_od"] = proxy_qc["oc_area_512"] < proxy_qc["od_area_512"]
proxy_qc["valid_cdr_range"] = proxy_qc["area_cdr_proxy"].between(0, 1, inclusive="neither")

proxy_qc["strict_anatomical_proxy_valid"] = (
    proxy_qc["valid_od_nonempty"] &
    proxy_qc["valid_oc_nonempty"] &
    proxy_qc["valid_oc_smaller_than_od"] &
    proxy_qc["valid_cdr_range"] &
    proxy_qc["mask_exists"]
)

proxy_qc["anatomical_proxy_qc_reason"] = proxy_qc.apply(assign_proxy_qc_reason, axis=1)

proxy_qc["anatomical_proxy_qc_status"] = np.where(
    proxy_qc["strict_anatomical_proxy_valid"],
    "valid_strict",
    "failed_anatomical_proxy",
)

valid_proxy = proxy_qc[proxy_qc["strict_anatomical_proxy_valid"]].copy()

if len(valid_proxy) > 0:
    od_area_p01 = float(valid_proxy["od_area_512"].quantile(0.01))
    od_area_p99 = float(valid_proxy["od_area_512"].quantile(0.99))
    cdr_p01 = float(valid_proxy["area_cdr_proxy"].quantile(0.01))
    cdr_p99 = float(valid_proxy["area_cdr_proxy"].quantile(0.99))

    proxy_qc["low_confidence_anatomical_proxy"] = (
        proxy_qc["strict_anatomical_proxy_valid"] &
        (
            (proxy_qc["od_area_512"] < od_area_p01) |
            (proxy_qc["od_area_512"] > od_area_p99) |
            (proxy_qc["area_cdr_proxy"] < cdr_p01) |
            (proxy_qc["area_cdr_proxy"] > cdr_p99)
        )
    )
else:
    proxy_qc["low_confidence_anatomical_proxy"] = False

failed_proxy_cases = proxy_qc[
    proxy_qc["anatomical_proxy_qc_status"] == "failed_anatomical_proxy"
].copy().reset_index(drop=True)

low_confidence_proxy_cases = proxy_qc[
    proxy_qc["low_confidence_anatomical_proxy"]
].copy().reset_index(drop=True)

repair_log = pd.DataFrame(repair_log_rows)

proxy_qc.to_csv(FINAL_PROXY_CSV, index=False)
proxy_qc.to_csv(FINAL_PROXY_WITH_QC_CSV, index=False)
proxy_qc.to_csv(RESULTS_PROXY_CSV, index=False)
proxy_qc.to_csv(RESULTS_PROXY_WITH_QC_CSV, index=False)
proxy_qc.to_csv(LIVE_PROXY_CSV, index=False)

failed_proxy_cases.to_csv(FAILED_PROXY_CASES_CSV, index=False)
low_confidence_proxy_cases.to_csv(LOW_CONFIDENCE_PROXY_CASES_CSV, index=False)
repair_log.to_csv(REPAIR_LOG_CSV, index=False)

elapsed_min = (time.time() - start_time) / 60.0

n_rows = len(proxy_qc)
n_valid = int(proxy_qc["strict_anatomical_proxy_valid"].sum())
n_failed = int((~proxy_qc["strict_anatomical_proxy_valid"]).sum())
n_low_confidence = int(proxy_qc["low_confidence_anatomical_proxy"].sum())
missing_masks = int((~proxy_qc["mask_exists"]).sum())

if SAVE_OVERLAYS:
    missing_overlays = int((~proxy_qc["overlay_exists"]).sum())
else:
    missing_overlays = 0

repair_attempts = int(len(repair_log))
repair_successes = int(repair_log["repair_applied"].sum()) if len(repair_log) > 0 else 0

print(f"Final proxy rows: {n_rows}")
print(f"Strict valid anatomical proxies: {n_valid}")
print(f"Failed anatomical proxies: {n_failed}")
print(f"Low-confidence valid proxies: {n_low_confidence}")
print(f"Repair attempts: {repair_attempts}")
print(f"Repair successes: {repair_successes}")
print(f"Missing masks: {missing_masks}")
print(f"Missing overlays: {missing_overlays}")
print(f"Elapsed minutes: {elapsed_min:.2f}")
print(f"Final proxy manifest: {FINAL_PROXY_WITH_QC_CSV}")
print(f"Failed proxy cases: {FAILED_PROXY_CASES_CSV}")
print(f"Repair log: {REPAIR_LOG_CSV}")

if len(failed_proxy_cases) > 0:
    print("\nFailed anatomical proxy cases:")
    print(
        failed_proxy_cases[
            [
                "sample_id",
                "od_area_512",
                "oc_area_512",
                "area_cdr_proxy",
                "anatomical_proxy_qc_reason",
                "mask_512_path",
                "overlay_512_path",
            ]
        ].to_string(index=False)
    )

if n_rows != 747:
    raise RuntimeError(f"Expected 747 HYGD proxy rows, found {n_rows}")

if missing_masks > 0:
    raise RuntimeError(f"Missing masks found: {missing_masks}")

if SAVE_OVERLAYS and missing_overlays > 0:
    raise RuntimeError(f"Missing overlays found: {missing_overlays}")

if n_valid < MIN_STRICT_VALID_PROXIES:
    raise RuntimeError(f"Too few strict valid anatomical proxies: {n_valid}/747")

print("Status: PASS")

In [ ]:
# Cell 5: Summarize HYGD anatomical proxy quality and distributions

FINAL_PROXY_WITH_QC_CSV = HYGD_PROXY_ROOT / "hygd_final_od_oc_proxy_manifest_with_qc.csv"
FAILED_PROXY_CASES_CSV = RESULTS_06_DIR / "notebook06_failed_anatomical_proxy_cases.csv"
LOW_CONFIDENCE_PROXY_CASES_CSV = RESULTS_06_DIR / "notebook06_low_confidence_anatomical_proxy_cases.csv"

if not FINAL_PROXY_WITH_QC_CSV.exists():
    raise FileNotFoundError(f"Missing final proxy with QC: {FINAL_PROXY_WITH_QC_CSV}")

proxy_qc = pd.read_csv(FINAL_PROXY_WITH_QC_CSV)
proxy_qc["sample_id"] = proxy_qc["sample_id"].astype(str)

strict_valid_proxy = proxy_qc[
    proxy_qc["strict_anatomical_proxy_valid"]
].copy().reset_index(drop=True)

failed_proxy_cases = proxy_qc[
    proxy_qc["anatomical_proxy_qc_status"] == "failed_anatomical_proxy"
].copy().reset_index(drop=True)

low_confidence_proxy_cases = proxy_qc[
    proxy_qc["low_confidence_anatomical_proxy"]
].copy().reset_index(drop=True)

summary_rows = []

for subset_name, subset_df in [
    ("all_cases", proxy_qc),
    ("strict_valid", strict_valid_proxy),
    ("low_confidence_valid", low_confidence_proxy_cases),
    ("failed_anatomical_proxy", failed_proxy_cases),
]:
    row = {
        "subset": subset_name,
        "n_cases": int(len(subset_df)),
    }

    for col in [
        "od_area_512",
        "oc_area_512",
        "rim_area_512",
        "area_cdr_proxy",
        "cup_to_disc_area_ratio",
        "od_confidence_mean",
        "oc_confidence_mean",
        "oc_od_centroid_distance_norm_od_radius",
    ]:
        if col in subset_df.columns and len(subset_df) > 0:
            values = pd.to_numeric(subset_df[col], errors="coerce")
            row[f"{col}_mean"] = float(values.mean())
            row[f"{col}_std"] = float(values.std())
            row[f"{col}_median"] = float(values.median())
            row[f"{col}_p01"] = float(values.quantile(0.01))
            row[f"{col}_p05"] = float(values.quantile(0.05))
            row[f"{col}_p95"] = float(values.quantile(0.95))
            row[f"{col}_p99"] = float(values.quantile(0.99))
        else:
            row[f"{col}_mean"] = np.nan
            row[f"{col}_std"] = np.nan
            row[f"{col}_median"] = np.nan
            row[f"{col}_p01"] = np.nan
            row[f"{col}_p05"] = np.nan
            row[f"{col}_p95"] = np.nan
            row[f"{col}_p99"] = np.nan

    summary_rows.append(row)

proxy_distribution_summary = pd.DataFrame(summary_rows)

qc_status_summary = (
    proxy_qc.groupby("anatomical_proxy_qc_status")
    .agg(
        n_cases=("sample_id", "count"),
        mean_od_area=("od_area_512", "mean"),
        mean_oc_area=("oc_area_512", "mean"),
        mean_area_cdr_proxy=("area_cdr_proxy", "mean"),
    )
    .reset_index()
)

low_confidence_reason_rows = []

if len(strict_valid_proxy) > 0:
    od_area_p01 = float(strict_valid_proxy["od_area_512"].quantile(0.01))
    od_area_p99 = float(strict_valid_proxy["od_area_512"].quantile(0.99))
    cdr_p01 = float(strict_valid_proxy["area_cdr_proxy"].quantile(0.01))
    cdr_p99 = float(strict_valid_proxy["area_cdr_proxy"].quantile(0.99))

    for _, row in low_confidence_proxy_cases.iterrows():
        reasons = []

        if row["od_area_512"] < od_area_p01:
            reasons.append("very_small_od_area")

        if row["od_area_512"] > od_area_p99:
            reasons.append("very_large_od_area")

        if row["area_cdr_proxy"] < cdr_p01:
            reasons.append("very_low_cdr_proxy")

        if row["area_cdr_proxy"] > cdr_p99:
            reasons.append("very_high_cdr_proxy")

        low_confidence_reason_rows.append(
            {
                "sample_id": row["sample_id"],
                "od_area_512": row["od_area_512"],
                "oc_area_512": row["oc_area_512"],
                "area_cdr_proxy": row["area_cdr_proxy"],
                "od_confidence_mean": row["od_confidence_mean"],
                "oc_confidence_mean": row["oc_confidence_mean"],
                "low_confidence_reasons": "|".join(reasons) if reasons else "low_confidence_unspecified",
                "mask_512_path": row["mask_512_path"],
                "overlay_512_path": row["overlay_512_path"],
            }
        )

low_confidence_reason_df = pd.DataFrame(low_confidence_reason_rows)

proxy_distribution_summary_csv = RESULTS_06_DIR / "notebook06_proxy_distribution_summary.csv"
qc_status_summary_csv = RESULTS_06_DIR / "notebook06_proxy_qc_status_summary.csv"
low_confidence_reason_csv = RESULTS_06_DIR / "notebook06_low_confidence_proxy_reasons.csv"

proxy_distribution_summary.to_csv(proxy_distribution_summary_csv, index=False)
qc_status_summary.to_csv(qc_status_summary_csv, index=False)
low_confidence_reason_df.to_csv(low_confidence_reason_csv, index=False)

print("QC status summary:")
print(qc_status_summary.to_string(index=False))

print("\nStrict-valid proxy distribution:")
print(
    proxy_distribution_summary[
        proxy_distribution_summary["subset"] == "strict_valid"
    ][
        [
            "subset",
            "n_cases",
            "od_area_512_mean",
            "oc_area_512_mean",
            "area_cdr_proxy_mean",
            "area_cdr_proxy_median",
            "od_confidence_mean_mean",
            "oc_confidence_mean_mean",
        ]
    ].to_string(index=False)
)

print(f"\nFailed cases: {len(failed_proxy_cases)}")
if len(failed_proxy_cases) > 0:
    print(
        failed_proxy_cases[
            [
                "sample_id",
                "od_area_512",
                "oc_area_512",
                "area_cdr_proxy",
                "anatomical_proxy_qc_reason",
            ]
        ].to_string(index=False)
    )

print(f"\nLow-confidence valid cases: {len(low_confidence_proxy_cases)}")
print(f"Distribution summary: {proxy_distribution_summary_csv}")
print(f"QC status summary: {qc_status_summary_csv}")
print(f"Low-confidence reasons: {low_confidence_reason_csv}")

if len(proxy_qc) != 747:
    raise RuntimeError(f"Expected 747 proxy rows, found {len(proxy_qc)}")

if len(strict_valid_proxy) != 745:
    raise RuntimeError(f"Expected 745 strict-valid proxies, found {len(strict_valid_proxy)}")

if len(failed_proxy_cases) != 2:
    raise RuntimeError(f"Expected 2 failed anatomical proxies, found {len(failed_proxy_cases)}")

print("Status: PASS")

In [ ]:
# Cell 6: Build visual audit panels for failed, low-confidence, and valid anatomical proxies

import matplotlib.pyplot as plt

FINAL_PROXY_WITH_QC_CSV = HYGD_PROXY_ROOT / "hygd_final_od_oc_proxy_manifest_with_qc.csv"
LOW_CONFIDENCE_REASON_CSV = RESULTS_06_DIR / "notebook06_low_confidence_proxy_reasons.csv"

if not FINAL_PROXY_WITH_QC_CSV.exists():
    raise FileNotFoundError(f"Missing final proxy with QC: {FINAL_PROXY_WITH_QC_CSV}")

proxy_qc = pd.read_csv(FINAL_PROXY_WITH_QC_CSV)
proxy_qc["sample_id"] = proxy_qc["sample_id"].astype(str)

if LOW_CONFIDENCE_REASON_CSV.exists():
    low_confidence_reasons = pd.read_csv(LOW_CONFIDENCE_REASON_CSV)
    low_confidence_reasons["sample_id"] = low_confidence_reasons["sample_id"].astype(str)
else:
    low_confidence_reasons = pd.DataFrame(columns=["sample_id", "low_confidence_reasons"])

proxy_qc = proxy_qc.merge(
    low_confidence_reasons[["sample_id", "low_confidence_reasons"]],
    on="sample_id",
    how="left",
)

failed_cases = proxy_qc[
    proxy_qc["anatomical_proxy_qc_status"] == "failed_anatomical_proxy"
].copy()

low_confidence_cases = proxy_qc[
    proxy_qc["low_confidence_anatomical_proxy"]
].copy()

valid_cases = proxy_qc[
    proxy_qc["strict_anatomical_proxy_valid"] &
    (~proxy_qc["low_confidence_anatomical_proxy"])
].copy()

valid_sample_cases = (
    valid_cases
    .sample(n=min(12, len(valid_cases)), random_state=int(PROJECT_CONFIG["seed"]))
    .copy()
)

low_confidence_sample_cases = (
    low_confidence_cases
    .sort_values(["area_cdr_proxy", "od_area_512"], ascending=[False, True])
    .head(12)
    .copy()
)

visual_audit_cases = pd.concat(
    [
        failed_cases.assign(audit_group="failed_anatomical_proxy"),
        low_confidence_sample_cases.assign(audit_group="low_confidence_valid"),
        valid_sample_cases.assign(audit_group="strict_valid_reference"),
    ],
    axis=0,
    ignore_index=True,
)

visual_audit_cases = visual_audit_cases.drop_duplicates(
    subset=["sample_id"],
    keep="first",
).reset_index(drop=True)

visual_audit_cases_csv = RESULTS_06_DIR / "notebook06_visual_audit_cases.csv"
visual_audit_cases.to_csv(visual_audit_cases_csv, index=False)


def make_overlay_panel(case_df, title, output_path, n_cols=4):
    if len(case_df) == 0:
        return False

    n_cases = len(case_df)
    n_rows = int(np.ceil(n_cases / n_cols))

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(4.5 * n_cols, 4.8 * n_rows),
    )

    axes = np.array(axes).reshape(-1)

    for ax_idx, ax in enumerate(axes):
        ax.axis("off")

        if ax_idx >= n_cases:
            continue

        row = case_df.iloc[ax_idx]
        overlay_path = Path(row["overlay_512_path"])

        if overlay_path.exists():
            overlay_bgr = cv2.imread(str(overlay_path), cv2.IMREAD_COLOR)
            overlay_rgb = cv2.cvtColor(overlay_bgr, cv2.COLOR_BGR2RGB)
            ax.imshow(overlay_rgb)
        else:
            ax.text(0.5, 0.5, "Missing overlay", ha="center", va="center")

        cdr = row["area_cdr_proxy"]
        cdr_text = "NaN" if pd.isna(cdr) else f"{cdr:.3f}"

        reason = row.get("anatomical_proxy_qc_reason", "valid")
        low_reason = row.get("low_confidence_reasons", "")

        if pd.isna(low_reason):
            low_reason = ""

        ax.set_title(
            f"{row['sample_id']}\n"
            f"OD={int(row['od_area_512'])}, OC={int(row['oc_area_512'])}, CDR={cdr_text}\n"
            f"{reason if reason != 'valid' else low_reason}",
            fontsize=9,
        )

    fig.suptitle(title, fontsize=14)
    fig.tight_layout()
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.close(fig)

    return True


failed_panel_path = RESULTS_06_DIR / "notebook06_failed_proxy_visual_panel.png"
low_confidence_panel_path = RESULTS_06_DIR / "notebook06_low_confidence_proxy_visual_panel.png"
valid_reference_panel_path = RESULTS_06_DIR / "notebook06_valid_proxy_reference_visual_panel.png"
combined_panel_path = RESULTS_06_DIR / "notebook06_combined_visual_audit_panel.png"

failed_panel_created = make_overlay_panel(
    failed_cases,
    "Failed anatomical proxy cases",
    failed_panel_path,
    n_cols=2,
)

low_confidence_panel_created = make_overlay_panel(
    low_confidence_sample_cases,
    "Low-confidence valid anatomical proxy cases",
    low_confidence_panel_path,
    n_cols=4,
)

valid_panel_created = make_overlay_panel(
    valid_sample_cases,
    "Strict-valid reference anatomical proxy cases",
    valid_reference_panel_path,
    n_cols=4,
)

combined_panel_created = make_overlay_panel(
    visual_audit_cases,
    "HYGD OD/OC anatomical proxy visual audit",
    combined_panel_path,
    n_cols=4,
)

visual_audit_summary = {
    "n_failed_cases": int(len(failed_cases)),
    "n_low_confidence_cases_total": int(len(low_confidence_cases)),
    "n_low_confidence_cases_panel": int(len(low_confidence_sample_cases)),
    "n_valid_reference_cases_panel": int(len(valid_sample_cases)),
    "n_visual_audit_cases_total": int(len(visual_audit_cases)),
    "failed_panel_created": bool(failed_panel_created),
    "low_confidence_panel_created": bool(low_confidence_panel_created),
    "valid_reference_panel_created": bool(valid_panel_created),
    "combined_panel_created": bool(combined_panel_created),
    "visual_audit_cases_csv": str(visual_audit_cases_csv),
    "failed_panel_path": str(failed_panel_path),
    "low_confidence_panel_path": str(low_confidence_panel_path),
    "valid_reference_panel_path": str(valid_reference_panel_path),
    "combined_panel_path": str(combined_panel_path),
}

visual_audit_summary_json = RESULTS_06_DIR / "notebook06_visual_audit_summary.json"

with open(visual_audit_summary_json, "w", encoding="utf-8") as f:
    json.dump(visual_audit_summary, f, indent=2)

print(f"Failed cases: {len(failed_cases)}")
print(f"Low-confidence valid cases: {len(low_confidence_cases)}")
print(f"Low-confidence cases in panel: {len(low_confidence_sample_cases)}")
print(f"Valid reference cases in panel: {len(valid_sample_cases)}")
print(f"Visual audit cases CSV: {visual_audit_cases_csv}")
print(f"Failed panel: {failed_panel_path}")
print(f"Low-confidence panel: {low_confidence_panel_path}")
print(f"Valid reference panel: {valid_reference_panel_path}")
print(f"Combined panel: {combined_panel_path}")
print(f"Visual audit summary: {visual_audit_summary_json}")

if len(failed_cases) != 2:
    raise RuntimeError(f"Expected 2 failed anatomical proxy cases, found {len(failed_cases)}")

if len(low_confidence_cases) != 29:
    raise RuntimeError(f"Expected 29 low-confidence valid cases, found {len(low_confidence_cases)}")

if not combined_panel_path.exists():
    raise RuntimeError("Combined visual audit panel was not saved.")

print("Status: PASS")

In [ ]:
# Cell 7: Build HYGD OD/OC proxy analysis manifest with metadata

FINAL_PROXY_WITH_QC_CSV = HYGD_PROXY_ROOT / "hygd_final_od_oc_proxy_manifest_with_qc.csv"

ANALYSIS_MANIFEST_CSV = HYGD_PROXY_ROOT / "hygd_final_od_oc_proxy_analysis_manifest.csv"
RESULTS_ANALYSIS_MANIFEST_CSV = RESULTS_06_DIR / "notebook06_hygd_final_od_oc_proxy_analysis_manifest.csv"
ANALYSIS_SUMMARY_CSV = RESULTS_06_DIR / "notebook06_hygd_proxy_analysis_manifest_summary.csv"
LABEL_QC_SUMMARY_CSV = RESULTS_06_DIR / "notebook06_hygd_proxy_qc_by_label_summary.csv"

if not FINAL_PROXY_WITH_QC_CSV.exists():
    raise FileNotFoundError(f"Missing final proxy with QC: {FINAL_PROXY_WITH_QC_CSV}")

proxy_qc = pd.read_csv(FINAL_PROXY_WITH_QC_CSV)
proxy_qc["sample_id"] = proxy_qc["sample_id"].astype(str)

hygd_metadata_analysis = hygd_metadata.copy().reset_index(drop=True)

hygd_metadata_analysis["sample_id"] = [
    get_hygd_sample_id(row, idx)
    for idx, row in hygd_metadata_analysis.iterrows()
]

hygd_metadata_analysis["sample_id"] = hygd_metadata_analysis["sample_id"].astype(str)

duplicate_proxy_ids = int(proxy_qc["sample_id"].duplicated().sum())
duplicate_metadata_ids = int(hygd_metadata_analysis["sample_id"].duplicated().sum())

if duplicate_proxy_ids > 0:
    raise RuntimeError(f"Duplicate proxy sample IDs found: {duplicate_proxy_ids}")

if duplicate_metadata_ids > 0:
    raise RuntimeError(f"Duplicate metadata sample IDs found: {duplicate_metadata_ids}")


def find_first_existing_column(dataframe, candidates):
    for col in candidates:
        if col in dataframe.columns:
            return col
    return None


patient_col = find_first_existing_column(
    hygd_metadata_analysis,
    [
        "patient_id",
        "Patient ID",
        "PatientID",
        "patient",
        "Patient",
        "id_patient",
        "subject_id",
        "Subject ID",
    ],
)

label_col = find_first_existing_column(
    hygd_metadata_analysis,
    [
        "label",
        "Label",
        "diagnosis",
        "Diagnosis",
        "class",
        "Class",
        "target",
        "Target",
        "GON",
        "gon",
        "glaucoma",
        "Glaucoma",
        "glaucoma_label",
        "binary_label",
        "is_glaucoma",
    ],
)

quality_col = find_first_existing_column(
    hygd_metadata_analysis,
    [
        "quality_score",
        "Quality Score",
        "quality",
        "Quality",
        "image_quality",
        "Image Quality",
    ],
)

metadata_keep_cols = ["sample_id"]

for col in [patient_col, label_col, quality_col, "image_path_for_inference"]:
    if col is not None and col in hygd_metadata_analysis.columns and col not in metadata_keep_cols:
        metadata_keep_cols.append(col)

metadata_selected = hygd_metadata_analysis[metadata_keep_cols].copy()

if patient_col is not None:
    metadata_selected = metadata_selected.rename(columns={patient_col: "patient_id"})

if label_col is not None:
    metadata_selected = metadata_selected.rename(columns={label_col: "source_label"})

if quality_col is not None:
    metadata_selected = metadata_selected.rename(columns={quality_col: "quality_score"})

analysis_manifest = proxy_qc.merge(
    metadata_selected,
    on="sample_id",
    how="left",
    validate="one_to_one",
)

missing_metadata = int(analysis_manifest["image_path_for_inference"].isna().sum()) if "image_path_for_inference" in analysis_manifest.columns else 747

if missing_metadata > 0:
    raise RuntimeError(f"Proxy rows without metadata match: {missing_metadata}")


def normalize_gon_label(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, (int, float, np.integer, np.floating)):
        if int(value) in [0, 1]:
            return int(value)

    text = str(value).strip().lower()

    positive_values = {
        "1",
        "gon+",
        "gon +",
        "positive",
        "glaucoma",
        "glaucomatous",
        "yes",
        "true",
        "g",
    }

    negative_values = {
        "0",
        "gon-",
        "gon -",
        "negative",
        "normal",
        "non-glaucoma",
        "nonglaucoma",
        "no",
        "false",
        "n",
    }

    if text in positive_values:
        return 1

    if text in negative_values:
        return 0

    return np.nan


if "source_label" in analysis_manifest.columns:
    analysis_manifest["gon_binary"] = analysis_manifest["source_label"].apply(normalize_gon_label)
    analysis_manifest["gon_label"] = analysis_manifest["gon_binary"].map({0: "GON-", 1: "GON+"})
else:
    analysis_manifest["gon_binary"] = np.nan
    analysis_manifest["gon_label"] = np.nan

analysis_manifest["include_strict_anatomical_analysis"] = (
    analysis_manifest["strict_anatomical_proxy_valid"].astype(bool)
)

analysis_manifest["include_high_confidence_anatomical_analysis"] = (
    analysis_manifest["strict_anatomical_proxy_valid"].astype(bool) &
    (~analysis_manifest["low_confidence_anatomical_proxy"].astype(bool))
)

analysis_manifest["include_sensitivity_anatomical_analysis"] = (
    analysis_manifest["strict_anatomical_proxy_valid"].astype(bool)
)

analysis_manifest["excluded_from_anatomical_analysis"] = (
    ~analysis_manifest["strict_anatomical_proxy_valid"].astype(bool)
)

summary_rows = [
    {
        "metric": "total_images",
        "value": int(len(analysis_manifest)),
    },
    {
        "metric": "strict_anatomical_valid",
        "value": int(analysis_manifest["include_strict_anatomical_analysis"].sum()),
    },
    {
        "metric": "high_confidence_anatomical_valid",
        "value": int(analysis_manifest["include_high_confidence_anatomical_analysis"].sum()),
    },
    {
        "metric": "low_confidence_valid",
        "value": int(analysis_manifest["low_confidence_anatomical_proxy"].sum()),
    },
    {
        "metric": "failed_anatomical_proxy",
        "value": int(analysis_manifest["excluded_from_anatomical_analysis"].sum()),
    },
    {
        "metric": "metadata_patient_column_found",
        "value": str(patient_col),
    },
    {
        "metric": "metadata_label_column_found",
        "value": str(label_col),
    },
    {
        "metric": "metadata_quality_column_found",
        "value": str(quality_col),
    },
    {
        "metric": "gon_binary_available",
        "value": int(analysis_manifest["gon_binary"].notna().sum()),
    },
]

if "patient_id" in analysis_manifest.columns:
    summary_rows.append(
        {
            "metric": "n_patients",
            "value": int(analysis_manifest["patient_id"].nunique()),
        }
    )

analysis_manifest_summary = pd.DataFrame(summary_rows)

if analysis_manifest["gon_binary"].notna().any():
    label_qc_summary = (
        analysis_manifest.groupby(["gon_label", "anatomical_proxy_qc_status"])
        .agg(
            n_images=("sample_id", "count"),
            mean_area_cdr_proxy=("area_cdr_proxy", "mean"),
            mean_od_area_512=("od_area_512", "mean"),
            mean_oc_area_512=("oc_area_512", "mean"),
            mean_od_confidence=("od_confidence_mean", "mean"),
            mean_oc_confidence=("oc_confidence_mean", "mean"),
        )
        .reset_index()
    )
else:
    label_qc_summary = pd.DataFrame(
        columns=[
            "gon_label",
            "anatomical_proxy_qc_status",
            "n_images",
            "mean_area_cdr_proxy",
            "mean_od_area_512",
            "mean_oc_area_512",
            "mean_od_confidence",
            "mean_oc_confidence",
        ]
    )

analysis_manifest.to_csv(ANALYSIS_MANIFEST_CSV, index=False)
analysis_manifest.to_csv(RESULTS_ANALYSIS_MANIFEST_CSV, index=False)
analysis_manifest_summary.to_csv(ANALYSIS_SUMMARY_CSV, index=False)
label_qc_summary.to_csv(LABEL_QC_SUMMARY_CSV, index=False)

print("Analysis manifest summary:")
print(analysis_manifest_summary.to_string(index=False))

print("\nLabel/QC summary:")
if len(label_qc_summary) > 0:
    print(label_qc_summary.to_string(index=False))
else:
    print("No label column was mapped to gon_binary.")

print(f"\nAnalysis manifest: {ANALYSIS_MANIFEST_CSV}")
print(f"Results analysis manifest: {RESULTS_ANALYSIS_MANIFEST_CSV}")
print(f"Analysis summary: {ANALYSIS_SUMMARY_CSV}")
print(f"Label QC summary: {LABEL_QC_SUMMARY_CSV}")

if len(analysis_manifest) != 747:
    raise RuntimeError(f"Expected 747 analysis rows, found {len(analysis_manifest)}")

if int(analysis_manifest["include_strict_anatomical_analysis"].sum()) != 745:
    raise RuntimeError("Unexpected number of strict anatomical valid cases.")

if int(analysis_manifest["excluded_from_anatomical_analysis"].sum()) != 2:
    raise RuntimeError("Unexpected number of failed anatomical proxy cases.")

print("Status: PASS")

In [ ]:
# Cell 8: Build patient-level HYGD OD/OC anatomical proxy summary

ANALYSIS_MANIFEST_CSV = HYGD_PROXY_ROOT / "hygd_final_od_oc_proxy_analysis_manifest.csv"

PATIENT_PROXY_CSV = HYGD_PROXY_ROOT / "hygd_final_od_oc_proxy_patient_summary.csv"
RESULTS_PATIENT_PROXY_CSV = RESULTS_06_DIR / "notebook06_hygd_final_od_oc_proxy_patient_summary.csv"

IMAGE_LABEL_SUMMARY_CSV = RESULTS_06_DIR / "notebook06_image_level_proxy_by_label_summary.csv"
PATIENT_LABEL_SUMMARY_CSV = RESULTS_06_DIR / "notebook06_patient_level_proxy_by_label_summary.csv"
PATIENT_QC_SUMMARY_CSV = RESULTS_06_DIR / "notebook06_patient_level_proxy_qc_summary.csv"

if not ANALYSIS_MANIFEST_CSV.exists():
    raise FileNotFoundError(f"Missing analysis manifest: {ANALYSIS_MANIFEST_CSV}")

analysis_manifest = pd.read_csv(ANALYSIS_MANIFEST_CSV)
analysis_manifest["sample_id"] = analysis_manifest["sample_id"].astype(str)

if "patient_id" not in analysis_manifest.columns:
    raise ValueError("patient_id column is required for patient-level summary.")

if "gon_binary" not in analysis_manifest.columns:
    raise ValueError("gon_binary column is required for patient-level summary.")

analysis_manifest["patient_id"] = analysis_manifest["patient_id"].astype(str)

strict_valid_images = analysis_manifest[
    analysis_manifest["include_strict_anatomical_analysis"]
].copy()

high_confidence_images = analysis_manifest[
    analysis_manifest["include_high_confidence_anatomical_analysis"]
].copy()

patient_rows = []

for patient_id, patient_df in analysis_manifest.groupby("patient_id"):
    strict_df = patient_df[patient_df["include_strict_anatomical_analysis"]].copy()
    high_conf_df = patient_df[patient_df["include_high_confidence_anatomical_analysis"]].copy()
    failed_df = patient_df[patient_df["excluded_from_anatomical_analysis"]].copy()
    low_conf_df = patient_df[patient_df["low_confidence_anatomical_proxy"]].copy()

    label_values = sorted(patient_df["gon_binary"].dropna().astype(int).unique().tolist())

    if len(label_values) == 1:
        patient_gon_binary = label_values[0]
        mixed_label_patient = False
    elif len(label_values) == 0:
        patient_gon_binary = np.nan
        mixed_label_patient = False
    else:
        patient_gon_binary = np.nan
        mixed_label_patient = True

    def safe_mean(df, col):
        if len(df) == 0 or col not in df.columns:
            return np.nan
        return float(pd.to_numeric(df[col], errors="coerce").mean())

    def safe_std(df, col):
        if len(df) <= 1 or col not in df.columns:
            return np.nan
        return float(pd.to_numeric(df[col], errors="coerce").std())

    def safe_min(df, col):
        if len(df) == 0 or col not in df.columns:
            return np.nan
        return float(pd.to_numeric(df[col], errors="coerce").min())

    def safe_max(df, col):
        if len(df) == 0 or col not in df.columns:
            return np.nan
        return float(pd.to_numeric(df[col], errors="coerce").max())

    cdr_mean = safe_mean(strict_df, "area_cdr_proxy")
    cdr_std = safe_std(strict_df, "area_cdr_proxy")

    if np.isfinite(cdr_mean) and cdr_mean > 0 and np.isfinite(cdr_std):
        cdr_cv = float(cdr_std / cdr_mean)
    else:
        cdr_cv = np.nan

    patient_rows.append(
        {
            "patient_id": patient_id,
            "gon_binary_patient": patient_gon_binary,
            "gon_label_patient": "GON+" if patient_gon_binary == 1 else ("GON-" if patient_gon_binary == 0 else np.nan),
            "mixed_label_patient": bool(mixed_label_patient),
            "n_images": int(len(patient_df)),
            "n_strict_valid_images": int(len(strict_df)),
            "n_high_confidence_images": int(len(high_conf_df)),
            "n_low_confidence_images": int(len(low_conf_df)),
            "n_failed_proxy_images": int(len(failed_df)),
            "patient_has_failed_proxy": bool(len(failed_df) > 0),
            "patient_has_low_confidence_proxy": bool(len(low_conf_df) > 0),
            "strict_valid_fraction": float(len(strict_df) / len(patient_df)) if len(patient_df) > 0 else np.nan,
            "high_confidence_fraction": float(len(high_conf_df) / len(patient_df)) if len(patient_df) > 0 else np.nan,
            "area_cdr_proxy_mean_strict": cdr_mean,
            "area_cdr_proxy_std_strict": cdr_std,
            "area_cdr_proxy_min_strict": safe_min(strict_df, "area_cdr_proxy"),
            "area_cdr_proxy_max_strict": safe_max(strict_df, "area_cdr_proxy"),
            "area_cdr_proxy_cv_strict": cdr_cv,
            "area_cdr_proxy_mean_high_confidence": safe_mean(high_conf_df, "area_cdr_proxy"),
            "cup_to_disc_area_ratio_mean_strict": safe_mean(strict_df, "cup_to_disc_area_ratio"),
            "od_area_512_mean_strict": safe_mean(strict_df, "od_area_512"),
            "oc_area_512_mean_strict": safe_mean(strict_df, "oc_area_512"),
            "rim_area_512_mean_strict": safe_mean(strict_df, "rim_area_512"),
            "od_confidence_mean_strict": safe_mean(strict_df, "od_confidence_mean"),
            "oc_confidence_mean_strict": safe_mean(strict_df, "oc_confidence_mean"),
            "quality_score_mean": safe_mean(patient_df, "quality_score") if "quality_score" in patient_df.columns else np.nan,
            "quality_score_min": safe_min(patient_df, "quality_score") if "quality_score" in patient_df.columns else np.nan,
        }
    )

patient_proxy_summary = pd.DataFrame(patient_rows)

image_level_proxy_by_label = (
    strict_valid_images.groupby("gon_label")
    .agg(
        n_images=("sample_id", "count"),
        n_patients=("patient_id", "nunique"),
        mean_area_cdr_proxy=("area_cdr_proxy", "mean"),
        std_area_cdr_proxy=("area_cdr_proxy", "std"),
        median_area_cdr_proxy=("area_cdr_proxy", "median"),
        mean_cup_to_disc_area_ratio=("cup_to_disc_area_ratio", "mean"),
        mean_od_area_512=("od_area_512", "mean"),
        mean_oc_area_512=("oc_area_512", "mean"),
        mean_od_confidence=("od_confidence_mean", "mean"),
        mean_oc_confidence=("oc_confidence_mean", "mean"),
    )
    .reset_index()
)

patient_valid = patient_proxy_summary[
    patient_proxy_summary["n_strict_valid_images"] > 0
].copy()

patient_level_proxy_by_label = (
    patient_valid.groupby("gon_label_patient")
    .agg(
        n_patients=("patient_id", "count"),
        n_images=("n_images", "sum"),
        n_strict_valid_images=("n_strict_valid_images", "sum"),
        mean_area_cdr_proxy=("area_cdr_proxy_mean_strict", "mean"),
        std_area_cdr_proxy=("area_cdr_proxy_mean_strict", "std"),
        median_area_cdr_proxy=("area_cdr_proxy_mean_strict", "median"),
        mean_cup_to_disc_area_ratio=("cup_to_disc_area_ratio_mean_strict", "mean"),
        mean_od_area_512=("od_area_512_mean_strict", "mean"),
        mean_oc_area_512=("oc_area_512_mean_strict", "mean"),
        mean_quality_score=("quality_score_mean", "mean"),
    )
    .reset_index()
)

patient_level_qc_summary = pd.DataFrame(
    [
        {
            "metric": "n_patients",
            "value": int(patient_proxy_summary["patient_id"].nunique()),
        },
        {
            "metric": "patients_with_at_least_one_strict_valid_proxy",
            "value": int((patient_proxy_summary["n_strict_valid_images"] > 0).sum()),
        },
        {
            "metric": "patients_with_failed_proxy_image",
            "value": int(patient_proxy_summary["patient_has_failed_proxy"].sum()),
        },
        {
            "metric": "patients_with_low_confidence_proxy_image",
            "value": int(patient_proxy_summary["patient_has_low_confidence_proxy"].sum()),
        },
        {
            "metric": "mixed_label_patients",
            "value": int(patient_proxy_summary["mixed_label_patient"].sum()),
        },
        {
            "metric": "mean_strict_valid_fraction",
            "value": float(patient_proxy_summary["strict_valid_fraction"].mean()),
        },
        {
            "metric": "mean_high_confidence_fraction",
            "value": float(patient_proxy_summary["high_confidence_fraction"].mean()),
        },
    ]
)

patient_proxy_summary.to_csv(PATIENT_PROXY_CSV, index=False)
patient_proxy_summary.to_csv(RESULTS_PATIENT_PROXY_CSV, index=False)
image_level_proxy_by_label.to_csv(IMAGE_LABEL_SUMMARY_CSV, index=False)
patient_level_proxy_by_label.to_csv(PATIENT_LABEL_SUMMARY_CSV, index=False)
patient_level_qc_summary.to_csv(PATIENT_QC_SUMMARY_CSV, index=False)

print("Patient-level QC summary:")
print(patient_level_qc_summary.to_string(index=False))

print("\nImage-level proxy by label:")
print(image_level_proxy_by_label.to_string(index=False))

print("\nPatient-level proxy by label:")
print(patient_level_proxy_by_label.to_string(index=False))

print(f"\nPatient proxy summary: {PATIENT_PROXY_CSV}")
print(f"Results patient proxy summary: {RESULTS_PATIENT_PROXY_CSV}")
print(f"Image-level label summary: {IMAGE_LABEL_SUMMARY_CSV}")
print(f"Patient-level label summary: {PATIENT_LABEL_SUMMARY_CSV}")
print(f"Patient-level QC summary: {PATIENT_QC_SUMMARY_CSV}")

if len(patient_proxy_summary) != 288:
    raise RuntimeError(f"Expected 288 patients, found {len(patient_proxy_summary)}")

if int(patient_proxy_summary["mixed_label_patient"].sum()) != 0:
    raise RuntimeError("Mixed-label patients found.")

if int((patient_proxy_summary["n_strict_valid_images"] > 0).sum()) < 286:
    raise RuntimeError("Too few patients with at least one strict-valid anatomical proxy.")

print("Status: PASS")

In [ ]:
# Cell 9: Statistical comparison of anatomical proxy features by glaucoma label

from scipy import stats

ANALYSIS_MANIFEST_CSV = HYGD_PROXY_ROOT / "hygd_final_od_oc_proxy_analysis_manifest.csv"
PATIENT_PROXY_CSV = HYGD_PROXY_ROOT / "hygd_final_od_oc_proxy_patient_summary.csv"

IMAGE_STATS_CSV = RESULTS_06_DIR / "notebook06_image_level_anatomical_proxy_label_statistics.csv"
PATIENT_STATS_CSV = RESULTS_06_DIR / "notebook06_patient_level_anatomical_proxy_label_statistics.csv"
STATISTICAL_SUMMARY_JSON = RESULTS_06_DIR / "notebook06_anatomical_proxy_statistical_summary.json"

if not ANALYSIS_MANIFEST_CSV.exists():
    raise FileNotFoundError(f"Missing analysis manifest: {ANALYSIS_MANIFEST_CSV}")

if not PATIENT_PROXY_CSV.exists():
    raise FileNotFoundError(f"Missing patient proxy summary: {PATIENT_PROXY_CSV}")

analysis_manifest = pd.read_csv(ANALYSIS_MANIFEST_CSV)
patient_proxy_summary = pd.read_csv(PATIENT_PROXY_CSV)

rng = np.random.default_rng(int(PROJECT_CONFIG["seed"]))

N_BOOTSTRAP = 5000

image_level_metrics = [
    "area_cdr_proxy",
    "cup_to_disc_area_ratio",
    "od_area_512",
    "oc_area_512",
    "rim_area_512",
    "od_confidence_mean",
    "oc_confidence_mean",
    "quality_score",
]

patient_level_metrics = [
    "area_cdr_proxy_mean_strict",
    "cup_to_disc_area_ratio_mean_strict",
    "od_area_512_mean_strict",
    "oc_area_512_mean_strict",
    "rim_area_512_mean_strict",
    "od_confidence_mean_strict",
    "oc_confidence_mean_strict",
    "quality_score_mean",
]


def bootstrap_ci_difference(pos_values, neg_values, statistic="mean", n_bootstrap=N_BOOTSTRAP):
    pos_values = np.asarray(pos_values, dtype=float)
    neg_values = np.asarray(neg_values, dtype=float)

    pos_values = pos_values[np.isfinite(pos_values)]
    neg_values = neg_values[np.isfinite(neg_values)]

    if len(pos_values) == 0 or len(neg_values) == 0:
        return np.nan, np.nan

    boot_diffs = []

    for _ in range(n_bootstrap):
        pos_sample = rng.choice(pos_values, size=len(pos_values), replace=True)
        neg_sample = rng.choice(neg_values, size=len(neg_values), replace=True)

        if statistic == "mean":
            diff = np.mean(pos_sample) - np.mean(neg_sample)
        elif statistic == "median":
            diff = np.median(pos_sample) - np.median(neg_sample)
        else:
            raise ValueError(f"Unknown statistic: {statistic}")

        boot_diffs.append(diff)

    lower, upper = np.percentile(boot_diffs, [2.5, 97.5])

    return float(lower), float(upper)


def summarize_group_comparison(dataframe, label_col, positive_value, negative_value, metric_cols, level_name):
    rows = []

    for metric in metric_cols:
        if metric not in dataframe.columns:
            continue

        subset = dataframe[[label_col, metric]].copy()
        subset[metric] = pd.to_numeric(subset[metric], errors="coerce")
        subset = subset.dropna(subset=[label_col, metric])

        pos_values = subset.loc[subset[label_col] == positive_value, metric].dropna().astype(float).values
        neg_values = subset.loc[subset[label_col] == negative_value, metric].dropna().astype(float).values

        if len(pos_values) == 0 or len(neg_values) == 0:
            continue

        pos_mean = float(np.mean(pos_values))
        neg_mean = float(np.mean(neg_values))
        pos_median = float(np.median(pos_values))
        neg_median = float(np.median(neg_values))

        mean_diff = pos_mean - neg_mean
        median_diff = pos_median - neg_median

        mann_whitney = stats.mannwhitneyu(
            pos_values,
            neg_values,
            alternative="two-sided",
        )

        u_statistic = float(mann_whitney.statistic)
        p_value = float(mann_whitney.pvalue)

        rank_biserial = float((2.0 * u_statistic / (len(pos_values) * len(neg_values))) - 1.0)

        mean_ci_low, mean_ci_high = bootstrap_ci_difference(
            pos_values,
            neg_values,
            statistic="mean",
        )

        median_ci_low, median_ci_high = bootstrap_ci_difference(
            pos_values,
            neg_values,
            statistic="median",
        )

        rows.append(
            {
                "level": level_name,
                "metric": metric,
                "n_positive": int(len(pos_values)),
                "n_negative": int(len(neg_values)),
                "positive_mean": pos_mean,
                "negative_mean": neg_mean,
                "mean_difference_positive_minus_negative": float(mean_diff),
                "mean_difference_ci95_low": mean_ci_low,
                "mean_difference_ci95_high": mean_ci_high,
                "positive_median": pos_median,
                "negative_median": neg_median,
                "median_difference_positive_minus_negative": float(median_diff),
                "median_difference_ci95_low": median_ci_low,
                "median_difference_ci95_high": median_ci_high,
                "mann_whitney_u": u_statistic,
                "mann_whitney_p": p_value,
                "rank_biserial_effect_size": rank_biserial,
            }
        )

    return pd.DataFrame(rows)


image_stats_input = analysis_manifest[
    analysis_manifest["include_strict_anatomical_analysis"]
].copy()

patient_stats_input = patient_proxy_summary[
    patient_proxy_summary["n_strict_valid_images"] > 0
].copy()

image_level_statistics = summarize_group_comparison(
    dataframe=image_stats_input,
    label_col="gon_binary",
    positive_value=1,
    negative_value=0,
    metric_cols=image_level_metrics,
    level_name="image_level_strict_valid",
)

patient_level_statistics = summarize_group_comparison(
    dataframe=patient_stats_input,
    label_col="gon_binary_patient",
    positive_value=1,
    negative_value=0,
    metric_cols=patient_level_metrics,
    level_name="patient_level_strict_valid",
)

image_level_statistics.to_csv(IMAGE_STATS_CSV, index=False)
patient_level_statistics.to_csv(PATIENT_STATS_CSV, index=False)

primary_patient_metric = patient_level_statistics[
    patient_level_statistics["metric"] == "area_cdr_proxy_mean_strict"
]

primary_image_metric = image_level_statistics[
    image_level_statistics["metric"] == "area_cdr_proxy"
]

if len(primary_patient_metric) != 1:
    raise RuntimeError("Primary patient-level CDR proxy statistic was not computed.")

if len(primary_image_metric) != 1:
    raise RuntimeError("Primary image-level CDR proxy statistic was not computed.")

primary_patient_row = primary_patient_metric.iloc[0].to_dict()
primary_image_row = primary_image_metric.iloc[0].to_dict()

statistical_summary = {
    "primary_patient_level_metric": "area_cdr_proxy_mean_strict",
    "primary_patient_level_n_positive": int(primary_patient_row["n_positive"]),
    "primary_patient_level_n_negative": int(primary_patient_row["n_negative"]),
    "primary_patient_level_positive_mean": float(primary_patient_row["positive_mean"]),
    "primary_patient_level_negative_mean": float(primary_patient_row["negative_mean"]),
    "primary_patient_level_mean_difference": float(primary_patient_row["mean_difference_positive_minus_negative"]),
    "primary_patient_level_mean_difference_ci95": [
        float(primary_patient_row["mean_difference_ci95_low"]),
        float(primary_patient_row["mean_difference_ci95_high"]),
    ],
    "primary_patient_level_mann_whitney_p": float(primary_patient_row["mann_whitney_p"]),
    "primary_patient_level_rank_biserial": float(primary_patient_row["rank_biserial_effect_size"]),
    "secondary_image_level_metric": "area_cdr_proxy",
    "secondary_image_level_n_positive": int(primary_image_row["n_positive"]),
    "secondary_image_level_n_negative": int(primary_image_row["n_negative"]),
    "secondary_image_level_positive_mean": float(primary_image_row["positive_mean"]),
    "secondary_image_level_negative_mean": float(primary_image_row["negative_mean"]),
    "secondary_image_level_mean_difference": float(primary_image_row["mean_difference_positive_minus_negative"]),
    "secondary_image_level_mean_difference_ci95": [
        float(primary_image_row["mean_difference_ci95_low"]),
        float(primary_image_row["mean_difference_ci95_high"]),
    ],
    "secondary_image_level_mann_whitney_p": float(primary_image_row["mann_whitney_p"]),
    "secondary_image_level_rank_biserial": float(primary_image_row["rank_biserial_effect_size"]),
    "interpretation": (
        "The OD/OC anatomical proxy shows higher cup-to-disc proxy values in GON+ than GON- "
        "at both patient and image levels. Patient-level statistics are treated as primary "
        "because multiple images can belong to the same patient."
    ),
}

with open(STATISTICAL_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(statistical_summary, f, indent=2)

print("Primary patient-level anatomical proxy comparison:")
print(
    primary_patient_metric[
        [
            "metric",
            "n_positive",
            "n_negative",
            "positive_mean",
            "negative_mean",
            "mean_difference_positive_minus_negative",
            "mean_difference_ci95_low",
            "mean_difference_ci95_high",
            "mann_whitney_p",
            "rank_biserial_effect_size",
        ]
    ].to_string(index=False)
)

print("\nSecondary image-level anatomical proxy comparison:")
print(
    primary_image_metric[
        [
            "metric",
            "n_positive",
            "n_negative",
            "positive_mean",
            "negative_mean",
            "mean_difference_positive_minus_negative",
            "mean_difference_ci95_low",
            "mean_difference_ci95_high",
            "mann_whitney_p",
            "rank_biserial_effect_size",
        ]
    ].to_string(index=False)
)

print(f"\nImage-level statistics: {IMAGE_STATS_CSV}")
print(f"Patient-level statistics: {PATIENT_STATS_CSV}")
print(f"Statistical summary: {STATISTICAL_SUMMARY_JSON}")

if int(primary_patient_row["n_positive"]) != 186:
    raise RuntimeError("Unexpected number of GON+ patients.")

if int(primary_patient_row["n_negative"]) != 102:
    raise RuntimeError("Unexpected number of GON- patients.")

if primary_patient_row["mean_difference_positive_minus_negative"] <= 0:
    raise RuntimeError("Expected patient-level CDR proxy to be higher in GON+ than GON-.")

print("Status: PASS")

In [ ]:
# Cell 10: Generate and display Nature-style anatomical proxy figures

import json
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display
from matplotlib.gridspec import GridSpec

ANALYSIS_MANIFEST_CSV = HYGD_PROXY_ROOT / "hygd_final_od_oc_proxy_analysis_manifest.csv"
PATIENT_PROXY_CSV = HYGD_PROXY_ROOT / "hygd_final_od_oc_proxy_patient_summary.csv"
STATISTICAL_SUMMARY_JSON = RESULTS_06_DIR / "notebook06_anatomical_proxy_statistical_summary.json"

FIGURE_DIR = RESULTS_06_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

if not ANALYSIS_MANIFEST_CSV.exists():
    raise FileNotFoundError(f"Missing analysis manifest: {ANALYSIS_MANIFEST_CSV}")

if not PATIENT_PROXY_CSV.exists():
    raise FileNotFoundError(f"Missing patient proxy summary: {PATIENT_PROXY_CSV}")

if not STATISTICAL_SUMMARY_JSON.exists():
    raise FileNotFoundError(f"Missing statistical summary: {STATISTICAL_SUMMARY_JSON}")

# Delete previous CDR/quality figures from this notebook, but keep visual audit panels.
delete_patterns = [
    "notebook06_main_anatomical_proxy_figure.*",
    "notebook06_patient_level_area_cdr_proxy_by_label_boxplot.*",
    "notebook06_image_level_area_cdr_proxy_by_label_boxplot.*",
    "notebook06_patient_level_area_cdr_proxy_by_label_histogram.*",
    "notebook06_image_level_area_cdr_proxy_by_label_histogram.*",
    "notebook06_patient_level_quality_vs_area_cdr_proxy.*",
    "notebook06_image_level_quality_vs_area_cdr_proxy.*",
    "notebook06_supplementary_image_level_cdr_proxy.*",
    "notebook06_supplementary_quality_sensitivity.*",
]

deleted_figures = []

for pattern in delete_patterns:
    for path in FIGURE_DIR.glob(pattern):
        path.unlink()
        deleted_figures.append(str(path))

analysis_manifest = pd.read_csv(ANALYSIS_MANIFEST_CSV)
patient_proxy_summary = pd.read_csv(PATIENT_PROXY_CSV)

with open(STATISTICAL_SUMMARY_JSON, "r", encoding="utf-8") as f:
    statistical_summary = json.load(f)


def bool_series(series):
    if series.dtype == bool:
        return series
    return series.astype(str).str.lower().isin(["true", "1", "yes"])


analysis_manifest["include_strict_anatomical_analysis"] = bool_series(
    analysis_manifest["include_strict_anatomical_analysis"]
)

analysis_manifest["include_high_confidence_anatomical_analysis"] = bool_series(
    analysis_manifest["include_high_confidence_anatomical_analysis"]
)

analysis_manifest["low_confidence_anatomical_proxy"] = bool_series(
    analysis_manifest["low_confidence_anatomical_proxy"]
)

strict_images = analysis_manifest[
    analysis_manifest["include_strict_anatomical_analysis"]
].copy()

high_confidence_images = analysis_manifest[
    analysis_manifest["include_high_confidence_anatomical_analysis"]
].copy()

strict_patients = patient_proxy_summary[
    patient_proxy_summary["n_strict_valid_images"] > 0
].copy()

strict_images["gon_binary"] = strict_images["gon_binary"].astype(int)
high_confidence_images["gon_binary"] = high_confidence_images["gon_binary"].astype(int)
strict_patients["gon_binary_patient"] = strict_patients["gon_binary_patient"].astype(int)

SEED = int(PROJECT_CONFIG.get("seed", 42))

LABEL_ORDER = [0, 1]
LABEL_NAMES_SHORT = ["GON−", "GON+"]
PALETTE = {
    0: "#4C78A8",
    1: "#E45756",
    "qc_high": "#54A24B",
    "qc_low": "#F2CF5B",
    "qc_fail": "#B279A2",
    "gray": "#4D4D4D",
}

plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
        "figure.titlesize": 13,
        "axes.linewidth": 0.8,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    }
)


def format_axis(ax, grid_axis="y"):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.8)
    ax.spines["bottom"].set_linewidth(0.8)

    if grid_axis is not None:
        ax.grid(axis=grid_axis, alpha=0.18, linewidth=0.7)

    ax.tick_params(axis="both", width=0.8, length=3)


def save_figure_all_formats(fig, output_stem):
    output_stem = Path(output_stem)

    png_path = output_stem.with_suffix(".png")
    pdf_path = output_stem.with_suffix(".pdf")
    svg_path = output_stem.with_suffix(".svg")

    fig.savefig(png_path, dpi=450, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(svg_path, bbox_inches="tight")

    return {
        "png": str(png_path),
        "pdf": str(pdf_path),
        "svg": str(svg_path),
    }


def display_then_save(fig, output_stem):
    display(fig)
    saved_paths = save_figure_all_formats(fig, output_stem)
    plt.close(fig)
    return saved_paths


def label_with_n(dataframe, label_col):
    labels = []

    for label_value, label_name in zip(LABEL_ORDER, LABEL_NAMES_SHORT):
        n = int((dataframe[label_col] == label_value).sum())
        labels.append(f"{label_name}\n(n={n})")

    return labels


def grouped_values(dataframe, label_col, value_col):
    values = []

    for label_value in LABEL_ORDER:
        group_values = pd.to_numeric(
            dataframe.loc[dataframe[label_col] == label_value, value_col],
            errors="coerce",
        ).dropna().values

        values.append(group_values)

    return values


def ecdf(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    values = np.sort(values)

    if len(values) == 0:
        return values, values

    y = np.arange(1, len(values) + 1) / len(values)
    return values, y


def add_panel_label(ax, label):
    ax.text(
        -0.12,
        1.08,
        label,
        transform=ax.transAxes,
        fontsize=13,
        fontweight="bold",
        ha="left",
        va="top",
    )


def add_stats_box(ax, lines):
    ax.text(
        0.03,
        0.97,
        "\n".join(lines),
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=8.5,
        bbox={
            "boxstyle": "round,pad=0.30",
            "facecolor": "white",
            "edgecolor": "0.70",
            "linewidth": 0.8,
            "alpha": 0.95,
        },
    )


def select_representative_overlay(image_df, label_value):
    candidates = image_df[
        (image_df["gon_binary"] == label_value) &
        (image_df["overlay_512_path"].apply(lambda x: Path(str(x)).exists()))
    ].copy()

    if len(candidates) == 0:
        return None

    median_cdr = float(candidates["area_cdr_proxy"].median())
    candidates["distance_to_group_median_cdr"] = (
        candidates["area_cdr_proxy"] - median_cdr
    ).abs()

    return (
        candidates
        .sort_values(["distance_to_group_median_cdr", "od_confidence_mean", "oc_confidence_mean"], ascending=[True, False, False])
        .iloc[0]
    )


def plot_overlay_case(ax, case_row, panel_label, title):
    ax.axis("off")
    add_panel_label(ax, panel_label)

    if case_row is None:
        ax.text(0.5, 0.5, "No overlay available", ha="center", va="center")
        ax.set_title(title, fontsize=10)
        return

    overlay_path = Path(str(case_row["overlay_512_path"]))
    overlay_bgr = cv2.imread(str(overlay_path), cv2.IMREAD_COLOR)

    if overlay_bgr is None:
        ax.text(0.5, 0.5, "Unreadable overlay", ha="center", va="center")
        ax.set_title(title, fontsize=10)
        return

    overlay_rgb = cv2.cvtColor(overlay_bgr, cv2.COLOR_BGR2RGB)
    ax.imshow(overlay_rgb)

    cdr = float(case_row["area_cdr_proxy"])
    od_area = int(case_row["od_area_512"])
    oc_area = int(case_row["oc_area_512"])

    ax.set_title(
        f"{title}\n{case_row['sample_id']} | CDR={cdr:.3f} | OD={od_area}, OC={oc_area}",
        fontsize=9.5,
        pad=4,
    )


def plot_patient_box(ax):
    values = grouped_values(
        strict_patients,
        "gon_binary_patient",
        "area_cdr_proxy_mean_strict",
    )

    labels = label_with_n(strict_patients, "gon_binary_patient")

    box = ax.boxplot(
        values,
        tick_labels=labels,
        showfliers=False,
        widths=0.45,
        patch_artist=True,
        medianprops={"linewidth": 1.8, "color": "#222222"},
        boxprops={"linewidth": 1.2, "edgecolor": "#222222"},
        whiskerprops={"linewidth": 1.0, "color": "#222222"},
        capprops={"linewidth": 1.0, "color": "#222222"},
    )

    for patch, label_value in zip(box["boxes"], LABEL_ORDER):
        patch.set_facecolor(PALETTE[label_value])
        patch.set_alpha(0.22)

    rng = np.random.default_rng(SEED)

    for x_pos, (group_values, label_value) in enumerate(zip(values, LABEL_ORDER), start=1):
        jitter = rng.normal(loc=0.0, scale=0.040, size=len(group_values))

        ax.scatter(
            np.full(len(group_values), x_pos) + jitter,
            group_values,
            s=13,
            alpha=0.42,
            color=PALETTE[label_value],
            linewidths=0,
            rasterized=True,
        )

        mean_value = np.mean(group_values)
        ax.scatter(
            [x_pos],
            [mean_value],
            marker="D",
            s=46,
            color=PALETTE[label_value],
            edgecolors="#111111",
            linewidths=0.8,
            zorder=5,
        )

    ax.set_ylabel("Patient-level area CDR proxy")
    ax.set_title("Patient-level anatomical proxy", loc="left", fontsize=11)
    add_panel_label(ax, "C")

    add_stats_box(
        ax,
        [
            f"Δmean = {statistical_summary['primary_patient_level_mean_difference']:.3f}",
            "95% CI = ["
            f"{statistical_summary['primary_patient_level_mean_difference_ci95'][0]:.3f}, "
            f"{statistical_summary['primary_patient_level_mean_difference_ci95'][1]:.3f}]",
            f"p = {statistical_summary['primary_patient_level_mann_whitney_p']:.2e}",
        ],
    )

    format_axis(ax)


def plot_patient_ecdf(ax):
    values = grouped_values(
        strict_patients,
        "gon_binary_patient",
        "area_cdr_proxy_mean_strict",
    )

    labels = label_with_n(strict_patients, "gon_binary_patient")

    for group_values, label_value, label_text in zip(values, LABEL_ORDER, labels):
        x, y = ecdf(group_values)

        ax.step(
            x,
            y,
            where="post",
            linewidth=2.0,
            color=PALETTE[label_value],
            label=label_text.replace("\n", " "),
        )

    ax.set_xlabel("Patient-level area CDR proxy")
    ax.set_ylabel("Cumulative fraction")
    ax.set_title("Distributional separation", loc="left", fontsize=11)
    ax.legend(frameon=False, loc="lower right")
    add_panel_label(ax, "D")
    format_axis(ax)


def plot_qc_summary(ax):
    high_conf = int(analysis_manifest["include_high_confidence_anatomical_analysis"].sum())
    low_conf = int(analysis_manifest["low_confidence_anatomical_proxy"].sum())
    failed = int((~analysis_manifest["include_strict_anatomical_analysis"]).sum())

    qc_labels = [
        "High-confidence\nvalid",
        "Low-confidence\nvalid",
        "Failed\nproxy",
    ]

    qc_values = [high_conf, low_conf, failed]
    qc_colors = [PALETTE["qc_high"], PALETTE["qc_low"], PALETTE["qc_fail"]]

    bars = ax.bar(
        np.arange(len(qc_values)),
        qc_values,
        color=qc_colors,
        alpha=0.85,
        edgecolor="#222222",
        linewidth=0.7,
    )

    total = sum(qc_values)

    for bar, value in zip(bars, qc_values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            value + 9,
            f"{value}\n({100 * value / total:.1f}%)",
            ha="center",
            va="bottom",
            fontsize=9,
        )

    ax.set_xticks(np.arange(len(qc_values)))
    ax.set_xticklabels(qc_labels)
    ax.set_ylabel("Number of images")
    ax.set_ylim(0, max(qc_values) * 1.18)
    ax.set_title("Anatomical proxy quality control", loc="left", fontsize=11)
    add_panel_label(ax, "E")
    format_axis(ax)


def make_main_nature_style_figure():
    gon_negative_case = select_representative_overlay(high_confidence_images, 0)
    gon_positive_case = select_representative_overlay(high_confidence_images, 1)

    fig = plt.figure(figsize=(12.2, 7.4))
    gs = GridSpec(
        2,
        3,
        figure=fig,
        height_ratios=[1.0, 0.95],
        width_ratios=[1.05, 1.05, 1.15],
        hspace=0.36,
        wspace=0.24,
    )

    ax_a = fig.add_subplot(gs[0, 0])
    ax_b = fig.add_subplot(gs[0, 1])
    ax_c = fig.add_subplot(gs[0, 2])
    ax_d = fig.add_subplot(gs[1, 0:2])
    ax_e = fig.add_subplot(gs[1, 2])

    plot_overlay_case(ax_a, gon_negative_case, "A", "Representative GON− OD/OC proxy")
    plot_overlay_case(ax_b, gon_positive_case, "B", "Representative GON+ OD/OC proxy")
    plot_patient_box(ax_c)
    plot_patient_ecdf(ax_d)
    plot_qc_summary(ax_e)

    fig.suptitle(
        "HYGD anatomical OD/OC proxy validation using the final multi-source segmentor",
        y=1.02,
        fontsize=13,
        fontweight="bold",
    )

    fig.tight_layout()

    return fig


def make_supplementary_image_level_figure():
    fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.6), gridspec_kw={"wspace": 0.26})

    values = grouped_values(strict_images, "gon_binary", "area_cdr_proxy")
    labels = label_with_n(strict_images, "gon_binary")

    ax = axes[0]
    box = ax.boxplot(
        values,
        tick_labels=labels,
        showfliers=False,
        widths=0.45,
        patch_artist=True,
        medianprops={"linewidth": 1.8, "color": "#222222"},
        boxprops={"linewidth": 1.2, "edgecolor": "#222222"},
        whiskerprops={"linewidth": 1.0, "color": "#222222"},
        capprops={"linewidth": 1.0, "color": "#222222"},
    )

    for patch, label_value in zip(box["boxes"], LABEL_ORDER):
        patch.set_facecolor(PALETTE[label_value])
        patch.set_alpha(0.22)

    rng = np.random.default_rng(SEED)

    for x_pos, (group_values, label_value) in enumerate(zip(values, LABEL_ORDER), start=1):
        jitter = rng.normal(loc=0.0, scale=0.040, size=len(group_values))
        ax.scatter(
            np.full(len(group_values), x_pos) + jitter,
            group_values,
            s=9,
            alpha=0.28,
            color=PALETTE[label_value],
            linewidths=0,
            rasterized=True,
        )

        ax.scatter(
            [x_pos],
            [np.mean(group_values)],
            marker="D",
            s=42,
            color=PALETTE[label_value],
            edgecolors="#111111",
            linewidths=0.8,
            zorder=5,
        )

    ax.set_ylabel("Image-level area CDR proxy")
    ax.set_title("Image-level anatomical proxy", loc="left")
    add_panel_label(ax, "A")
    add_stats_box(
        ax,
        [
            f"Δmean = {statistical_summary['secondary_image_level_mean_difference']:.3f}",
            "95% CI = ["
            f"{statistical_summary['secondary_image_level_mean_difference_ci95'][0]:.3f}, "
            f"{statistical_summary['secondary_image_level_mean_difference_ci95'][1]:.3f}]",
            f"p = {statistical_summary['secondary_image_level_mann_whitney_p']:.2e}",
        ],
    )
    format_axis(ax)

    ax = axes[1]

    for group_values, label_value, label_text in zip(values, LABEL_ORDER, labels):
        x, y = ecdf(group_values)

        ax.step(
            x,
            y,
            where="post",
            linewidth=2.0,
            color=PALETTE[label_value],
            label=label_text.replace("\n", " "),
        )

    ax.set_xlabel("Image-level area CDR proxy")
    ax.set_ylabel("Cumulative fraction")
    ax.set_title("Image-level distributional separation", loc="left")
    ax.legend(frameon=False, loc="lower right")
    add_panel_label(ax, "B")
    format_axis(ax)

    fig.suptitle(
        "Supplementary image-level OD/OC anatomical proxy analysis",
        y=1.04,
        fontsize=12,
        fontweight="bold",
    )

    fig.tight_layout()

    return fig


def make_supplementary_quality_figure():
    fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.6), gridspec_kw={"wspace": 0.26})

    patient_plot_df = strict_patients[
        ["quality_score_mean", "area_cdr_proxy_mean_strict", "gon_binary_patient"]
    ].copy()

    patient_plot_df["quality_score_mean"] = pd.to_numeric(patient_plot_df["quality_score_mean"], errors="coerce")
    patient_plot_df["area_cdr_proxy_mean_strict"] = pd.to_numeric(patient_plot_df["area_cdr_proxy_mean_strict"], errors="coerce")
    patient_plot_df = patient_plot_df.dropna()

    image_plot_df = strict_images[
        ["quality_score", "area_cdr_proxy", "gon_binary"]
    ].copy()

    image_plot_df["quality_score"] = pd.to_numeric(image_plot_df["quality_score"], errors="coerce")
    image_plot_df["area_cdr_proxy"] = pd.to_numeric(image_plot_df["area_cdr_proxy"], errors="coerce")
    image_plot_df = image_plot_df.dropna()

    def scatter_quality(ax, dataframe, quality_col, cdr_col, label_col, title, panel_label):
        for label_value, label_name in zip(LABEL_ORDER, LABEL_NAMES_SHORT):
            subset = dataframe[dataframe[label_col] == label_value]

            ax.scatter(
                subset[quality_col],
                subset[cdr_col],
                s=16,
                alpha=0.45,
                color=PALETTE[label_value],
                label=label_name,
                linewidths=0,
                rasterized=True,
            )

        if len(dataframe) >= 3:
            x = dataframe[quality_col].values.astype(float)
            y = dataframe[cdr_col].values.astype(float)

            slope, intercept = np.polyfit(x, y, deg=1)
            x_line = np.linspace(np.nanmin(x), np.nanmax(x), 100)
            y_line = slope * x_line + intercept

            ax.plot(
                x_line,
                y_line,
                linewidth=1.8,
                linestyle="--",
                color="#333333",
            )

            rho, p_value = stats.spearmanr(x, y, nan_policy="omit")

            add_stats_box(
                ax,
                [
                    f"Spearman ρ = {rho:.3f}",
                    f"p = {p_value:.2e}",
                ],
            )

        ax.set_xlabel("Quality score")
        ax.set_ylabel("Area CDR proxy")
        ax.set_title(title, loc="left")
        ax.legend(frameon=False)
        add_panel_label(ax, panel_label)
        format_axis(ax)

    from scipy import stats

    scatter_quality(
        axes[0],
        patient_plot_df,
        "quality_score_mean",
        "area_cdr_proxy_mean_strict",
        "gon_binary_patient",
        "Patient-level quality sensitivity",
        "A",
    )

    scatter_quality(
        axes[1],
        image_plot_df,
        "quality_score",
        "area_cdr_proxy",
        "gon_binary",
        "Image-level quality sensitivity",
        "B",
    )

    fig.suptitle(
        "Supplementary quality sensitivity analysis",
        y=1.04,
        fontsize=12,
        fontweight="bold",
    )

    fig.tight_layout()

    return fig


main_figure_stem = FIGURE_DIR / "notebook06_main_anatomical_proxy_figure"
supp_image_stem = FIGURE_DIR / "notebook06_supplementary_image_level_cdr_proxy"
supp_quality_stem = FIGURE_DIR / "notebook06_supplementary_quality_sensitivity"

print(f"Deleted previous figures: {len(deleted_figures)}")

main_fig = make_main_nature_style_figure()
main_paths = display_then_save(main_fig, main_figure_stem)

supp_image_fig = make_supplementary_image_level_figure()
supp_image_paths = display_then_save(supp_image_fig, supp_image_stem)

supp_quality_fig = make_supplementary_quality_figure()
supp_quality_paths = display_then_save(supp_quality_fig, supp_quality_stem)

figure_records = []

for figure_name, paths in [
    ("main_anatomical_proxy_figure", main_paths),
    ("supplementary_image_level_cdr_proxy", supp_image_paths),
    ("supplementary_quality_sensitivity", supp_quality_paths),
]:
    for fmt, path in paths.items():
        figure_records.append(
            {
                "figure_name": figure_name,
                "format": fmt,
                "figure_path": path,
                "exists": Path(path).exists(),
            }
        )

figure_manifest = pd.DataFrame(figure_records)

figure_manifest_csv = RESULTS_06_DIR / "notebook06_anatomical_proxy_figure_manifest.csv"
figure_manifest.to_csv(figure_manifest_csv, index=False)

figure_summary = {
    "figure_style": "publication_focused_main_plus_supplementary",
    "deleted_previous_figures": len(deleted_figures),
    "main_figure_png": main_paths["png"],
    "main_figure_pdf": main_paths["pdf"],
    "main_figure_svg": main_paths["svg"],
    "supplementary_image_level_png": supp_image_paths["png"],
    "supplementary_quality_png": supp_quality_paths["png"],
    "patient_level_primary_difference": statistical_summary["primary_patient_level_mean_difference"],
    "patient_level_primary_ci95": statistical_summary["primary_patient_level_mean_difference_ci95"],
    "patient_level_primary_p": statistical_summary["primary_patient_level_mann_whitney_p"],
    "patient_level_primary_rank_biserial": statistical_summary["primary_patient_level_rank_biserial"],
    "image_level_secondary_difference": statistical_summary["secondary_image_level_mean_difference"],
    "image_level_secondary_ci95": statistical_summary["secondary_image_level_mean_difference_ci95"],
    "image_level_secondary_p": statistical_summary["secondary_image_level_mann_whitney_p"],
    "image_level_secondary_rank_biserial": statistical_summary["secondary_image_level_rank_biserial"],
    "figure_manifest": str(figure_manifest_csv),
}

figure_summary_json = RESULTS_06_DIR / "notebook06_anatomical_proxy_figure_summary.json"

with open(figure_summary_json, "w", encoding="utf-8") as f:
    json.dump(figure_summary, f, indent=2)

print("Generated and displayed figures:")
print(figure_manifest.to_string(index=False))

print("\nPrimary patient-level result:")
print(
    f"Difference GON+ - GON− = {statistical_summary['primary_patient_level_mean_difference']:.6f}, "
    f"95% CI = [{statistical_summary['primary_patient_level_mean_difference_ci95'][0]:.6f}, "
    f"{statistical_summary['primary_patient_level_mean_difference_ci95'][1]:.6f}], "
    f"p = {statistical_summary['primary_patient_level_mann_whitney_p']:.3e}"
)

print(f"\nFigure manifest: {figure_manifest_csv}")
print(f"Figure summary: {figure_summary_json}")

required_paths = [
    main_paths["png"],
    main_paths["pdf"],
    main_paths["svg"],
    supp_image_paths["png"],
    supp_image_paths["pdf"],
    supp_image_paths["svg"],
    supp_quality_paths["png"],
    supp_quality_paths["pdf"],
    supp_quality_paths["svg"],
]

missing_required_figures = [
    path for path in required_paths
    if not Path(path).exists()
]

if missing_required_figures:
    raise RuntimeError(f"Missing required figures: {missing_required_figures}")

print("Status: PASS")

In [ ]:
# Cell 10B: Regenerate final publication candidate main figure with cleaner layout

from matplotlib.gridspec import GridSpec
from IPython.display import display

MAIN_FIGURE_V2_STEM = FIGURE_DIR / "notebook06_main_anatomical_proxy_figure_v2"

# Remove previous main v2/final files to keep outputs clean.
for old_path in list(FIGURE_DIR.glob("notebook06_main_anatomical_proxy_figure_v2.*")):
    old_path.unlink()

for old_path in list(FIGURE_DIR.glob("notebook06_main_anatomical_proxy_figure_final.*")):
    old_path.unlink()

old_manifest_paths = [
    RESULTS_06_DIR / "notebook06_main_anatomical_proxy_figure_v2_manifest.csv",
    RESULTS_06_DIR / "notebook06_main_anatomical_proxy_figure_v2_summary.json",
    RESULTS_06_DIR / "notebook06_main_anatomical_proxy_figure_final_manifest.csv",
    RESULTS_06_DIR / "notebook06_main_anatomical_proxy_figure_final_summary.json",
]

for old_path in old_manifest_paths:
    if old_path.exists():
        old_path.unlink()


def select_high_quality_representative_overlay(image_df, label_value):
    candidates = image_df[
        (image_df["gon_binary"] == label_value) &
        (image_df["overlay_512_path"].apply(lambda x: Path(str(x)).exists()))
    ].copy()

    if len(candidates) == 0:
        return None

    if "quality_score" in candidates.columns:
        candidates["quality_score"] = pd.to_numeric(candidates["quality_score"], errors="coerce")
        quality_cutoff = candidates["quality_score"].quantile(0.65)
        high_quality = candidates[candidates["quality_score"] >= quality_cutoff].copy()

        if len(high_quality) >= 5:
            candidates = high_quality

    candidates["area_cdr_proxy"] = pd.to_numeric(candidates["area_cdr_proxy"], errors="coerce")
    candidates["od_confidence_mean"] = pd.to_numeric(candidates["od_confidence_mean"], errors="coerce")
    candidates["oc_confidence_mean"] = pd.to_numeric(candidates["oc_confidence_mean"], errors="coerce")

    median_cdr = float(candidates["area_cdr_proxy"].median())

    candidates["distance_to_group_median_cdr"] = (
        candidates["area_cdr_proxy"] - median_cdr
    ).abs()

    return (
        candidates
        .sort_values(
            ["distance_to_group_median_cdr", "od_confidence_mean", "oc_confidence_mean"],
            ascending=[True, False, False],
        )
        .iloc[0]
    )


def crop_around_od_overlay_v2(case_row, crop_size=190):
    overlay_path = Path(str(case_row["overlay_512_path"]))
    overlay_bgr = cv2.imread(str(overlay_path), cv2.IMREAD_COLOR)

    if overlay_bgr is None:
        return None

    overlay_rgb = cv2.cvtColor(overlay_bgr, cv2.COLOR_BGR2RGB)

    cx = float(case_row["od_centroid_x_512"])
    cy = float(case_row["od_centroid_y_512"])

    if not np.isfinite(cx) or not np.isfinite(cy):
        return overlay_rgb

    h, w = overlay_rgb.shape[:2]
    half = crop_size // 2

    x1 = max(0, int(round(cx)) - half)
    y1 = max(0, int(round(cy)) - half)
    x2 = min(w, x1 + crop_size)
    y2 = min(h, y1 + crop_size)

    if x2 - x1 < crop_size:
        x1 = max(0, x2 - crop_size)

    if y2 - y1 < crop_size:
        y1 = max(0, y2 - crop_size)

    return overlay_rgb[y1:y2, x1:x2]


def add_crop_panel_label(ax, label):
    ax.text(
        0.025,
        0.975,
        label,
        transform=ax.transAxes,
        fontsize=15,
        fontweight="bold",
        ha="left",
        va="top",
        color="black",
        bbox={
            "boxstyle": "round,pad=0.16",
            "facecolor": "white",
            "edgecolor": "none",
            "alpha": 0.78,
        },
    )


def add_plot_panel_label(ax, label):
    ax.text(
        -0.12,
        1.06,
        label,
        transform=ax.transAxes,
        fontsize=15,
        fontweight="bold",
        ha="left",
        va="bottom",
        clip_on=False,
    )


def plot_overlay_crop_case_v2(ax, case_row, panel_label, group_title):
    ax.axis("off")
    add_crop_panel_label(ax, panel_label)

    if case_row is None:
        ax.text(0.5, 0.5, "No overlay available", ha="center", va="center")
        ax.set_title(group_title, fontsize=10, pad=10)
        return

    crop_rgb = crop_around_od_overlay_v2(case_row, crop_size=190)

    if crop_rgb is None:
        ax.text(0.5, 0.5, "Unreadable overlay", ha="center", va="center")
        ax.set_title(group_title, fontsize=10, pad=10)
        return

    ax.imshow(crop_rgb)

    cdr = float(case_row["area_cdr_proxy"])
    od_area = int(case_row["od_area_512"])
    oc_area = int(case_row["oc_area_512"])

    ax.set_title(group_title, fontsize=10.5, pad=10)

    ax.text(
        0.03,
        0.04,
        f"{case_row['sample_id']}\nCDR={cdr:.3f} | OD={od_area} | OC={oc_area}",
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=8.2,
        color="white",
        bbox={
            "boxstyle": "round,pad=0.25",
            "facecolor": "black",
            "edgecolor": "none",
            "alpha": 0.58,
        },
    )


def plot_patient_box_v2(ax):
    values = grouped_values(
        strict_patients,
        "gon_binary_patient",
        "area_cdr_proxy_mean_strict",
    )

    labels = label_with_n(strict_patients, "gon_binary_patient")

    box = ax.boxplot(
        values,
        tick_labels=labels,
        showfliers=False,
        widths=0.45,
        patch_artist=True,
        medianprops={"linewidth": 1.8, "color": "#222222"},
        boxprops={"linewidth": 1.2, "edgecolor": "#222222"},
        whiskerprops={"linewidth": 1.0, "color": "#222222"},
        capprops={"linewidth": 1.0, "color": "#222222"},
    )

    for patch, label_value in zip(box["boxes"], LABEL_ORDER):
        patch.set_facecolor(PALETTE[label_value])
        patch.set_alpha(0.22)

    rng = np.random.default_rng(SEED)

    for x_pos, (group_values, label_value) in enumerate(zip(values, LABEL_ORDER), start=1):
        jitter = rng.normal(loc=0.0, scale=0.040, size=len(group_values))

        ax.scatter(
            np.full(len(group_values), x_pos) + jitter,
            group_values,
            s=13,
            alpha=0.42,
            color=PALETTE[label_value],
            linewidths=0,
            rasterized=True,
        )

        ax.scatter(
            [x_pos],
            [np.mean(group_values)],
            marker="D",
            s=46,
            color=PALETTE[label_value],
            edgecolors="#111111",
            linewidths=0.8,
            zorder=5,
        )

    ax.set_ylabel("Patient-level area CDR proxy")
    ax.set_title("Patient-level anatomical proxy", loc="left", fontsize=11)
    add_plot_panel_label(ax, "C")

    add_stats_box(
        ax,
        [
            f"Δmean = {statistical_summary['primary_patient_level_mean_difference']:.3f}",
            "95% CI = ["
            f"{statistical_summary['primary_patient_level_mean_difference_ci95'][0]:.3f}, "
            f"{statistical_summary['primary_patient_level_mean_difference_ci95'][1]:.3f}]",
            f"p = {statistical_summary['primary_patient_level_mann_whitney_p']:.2e}",
        ],
    )

    format_axis(ax)


def plot_patient_ecdf_v2(ax):
    values = grouped_values(
        strict_patients,
        "gon_binary_patient",
        "area_cdr_proxy_mean_strict",
    )

    labels = label_with_n(strict_patients, "gon_binary_patient")

    for group_values, label_value, label_text in zip(values, LABEL_ORDER, labels):
        x, y = ecdf(group_values)

        ax.step(
            x,
            y,
            where="post",
            linewidth=2.0,
            color=PALETTE[label_value],
            label=label_text.replace("\n", " "),
        )

    ax.set_xlabel("Patient-level area CDR proxy")
    ax.set_ylabel("Cumulative fraction")
    ax.set_title("Distributional separation", loc="left", fontsize=11)
    ax.legend(frameon=False, loc="lower right")
    add_plot_panel_label(ax, "D")
    format_axis(ax)


def plot_qc_summary_v2(ax):
    high_conf = int(analysis_manifest["include_high_confidence_anatomical_analysis"].sum())
    low_conf = int(analysis_manifest["low_confidence_anatomical_proxy"].sum())
    failed = int((~analysis_manifest["include_strict_anatomical_analysis"]).sum())

    qc_labels = [
        "High-confidence\nvalid",
        "Low-confidence\nvalid",
        "Failed\nproxy",
    ]

    qc_values = [high_conf, low_conf, failed]
    qc_colors = [PALETTE["qc_high"], PALETTE["qc_low"], PALETTE["qc_fail"]]

    bars = ax.bar(
        np.arange(len(qc_values)),
        qc_values,
        color=qc_colors,
        alpha=0.85,
        edgecolor="#222222",
        linewidth=0.7,
    )

    total = sum(qc_values)

    for bar, value in zip(bars, qc_values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            value + 9,
            f"{value}\n({100 * value / total:.1f}%)",
            ha="center",
            va="bottom",
            fontsize=9,
        )

    ax.set_xticks(np.arange(len(qc_values)))
    ax.set_xticklabels(qc_labels)
    ax.set_ylabel("Number of images")
    ax.set_ylim(0, max(qc_values) * 1.18)
    ax.set_title("Anatomical proxy quality control", loc="left", fontsize=11)
    add_plot_panel_label(ax, "E")
    format_axis(ax)


def make_main_nature_style_figure_v2():
    gon_negative_case = select_high_quality_representative_overlay(high_confidence_images, 0)
    gon_positive_case = select_high_quality_representative_overlay(high_confidence_images, 1)

    fig = plt.figure(figsize=(12.0, 7.1))

    gs = GridSpec(
        2,
        3,
        figure=fig,
        height_ratios=[0.95, 0.95],
        width_ratios=[0.95, 0.95, 1.20],
        hspace=0.50,
        wspace=0.34,
    )

    ax_a = fig.add_subplot(gs[0, 0])
    ax_b = fig.add_subplot(gs[0, 1])
    ax_c = fig.add_subplot(gs[0, 2])
    ax_d = fig.add_subplot(gs[1, 0:2])
    ax_e = fig.add_subplot(gs[1, 2])

    plot_overlay_crop_case_v2(ax_a, gon_negative_case, "A", "Representative GON− OD/OC crop")
    plot_overlay_crop_case_v2(ax_b, gon_positive_case, "B", "Representative GON+ OD/OC crop")
    plot_patient_box_v2(ax_c)
    plot_patient_ecdf_v2(ax_d)
    plot_qc_summary_v2(ax_e)

    fig.suptitle(
        "HYGD anatomical OD/OC proxy validation using the final multi-source segmentor",
        y=0.965,
        fontsize=13,
        fontweight="bold",
    )

    fig.subplots_adjust(
        left=0.055,
        right=0.985,
        bottom=0.075,
        top=0.885,
        wspace=0.34,
        hspace=0.50,
    )

    return fig


main_fig_v2 = make_main_nature_style_figure_v2()

display(main_fig_v2)

main_v2_paths = save_figure_all_formats(main_fig_v2, MAIN_FIGURE_V2_STEM)
plt.close(main_fig_v2)

main_v2_manifest = pd.DataFrame(
    [
        {
            "figure_name": "main_anatomical_proxy_figure_v2",
            "format": fmt,
            "figure_path": path,
            "exists": Path(path).exists(),
        }
        for fmt, path in main_v2_paths.items()
    ]
)

main_v2_manifest_csv = RESULTS_06_DIR / "notebook06_main_anatomical_proxy_figure_v2_manifest.csv"
main_v2_summary_json = RESULTS_06_DIR / "notebook06_main_anatomical_proxy_figure_v2_summary.json"

main_v2_manifest.to_csv(main_v2_manifest_csv, index=False)

main_v2_summary = {
    "figure_style": "main_publication_candidate_high_quality_od_centered_crops",
    "main_figure_v2_png": main_v2_paths["png"],
    "main_figure_v2_pdf": main_v2_paths["pdf"],
    "main_figure_v2_svg": main_v2_paths["svg"],
    "reason_for_v2": "Uses high-quality representative OD-centered crops, clean panel labels, patient-level primary statistics, ECDF distribution, and QC summary.",
    "primary_patient_level_difference": statistical_summary["primary_patient_level_mean_difference"],
    "primary_patient_level_ci95": statistical_summary["primary_patient_level_mean_difference_ci95"],
    "primary_patient_level_p": statistical_summary["primary_patient_level_mann_whitney_p"],
}

with open(main_v2_summary_json, "w", encoding="utf-8") as f:
    json.dump(main_v2_summary, f, indent=2)

print("Generated main figure v2:")
print(main_v2_manifest.to_string(index=False))
print(f"Main figure v2 manifest: {main_v2_manifest_csv}")
print(f"Main figure v2 summary: {main_v2_summary_json}")

missing_v2 = [
    path for path in main_v2_paths.values()
    if not Path(path).exists()
]

if missing_v2:
    raise RuntimeError(f"Missing main figure v2 files: {missing_v2}")

print("Status: PASS")

In [ ]:
# Cell 11: Final status for Notebook 06

FINAL_STATUS_JSON = RESULTS_06_DIR / "notebook06_final_status.json"
FINAL_STATUS_TXT = RESULTS_06_DIR / "notebook06_FINAL_STATUS.txt"
FINAL_STATUS_CHECKS_CSV = RESULTS_06_DIR / "notebook06_final_status_checks.csv"


def as_bool_series(series):
    if series.dtype == bool:
        return series.fillna(False)

    return series.astype(str).str.lower().isin(["true", "1", "yes"])


required_files = {
    "setup_summary": RESULTS_06_DIR / "notebook06_setup_summary.json",
    "model_loading_smoke_test": RESULTS_06_DIR / "notebook06_model_loading_smoke_test.json",
    "one_sample_proxy_smoke_test": RESULTS_06_DIR / "notebook06_one_sample_anatomical_proxy_smoke_test.csv",
    "final_proxy_manifest_with_qc": HYGD_PROXY_ROOT / "hygd_final_od_oc_proxy_manifest_with_qc.csv",
    "analysis_manifest": HYGD_PROXY_ROOT / "hygd_final_od_oc_proxy_analysis_manifest.csv",
    "patient_proxy_summary": HYGD_PROXY_ROOT / "hygd_final_od_oc_proxy_patient_summary.csv",
    "failed_proxy_cases": RESULTS_06_DIR / "notebook06_failed_anatomical_proxy_cases.csv",
    "low_confidence_cases": RESULTS_06_DIR / "notebook06_low_confidence_anatomical_proxy_cases.csv",
    "visual_audit_summary": RESULTS_06_DIR / "notebook06_visual_audit_summary.json",
    "proxy_distribution_summary": RESULTS_06_DIR / "notebook06_proxy_distribution_summary.csv",
    "image_level_statistics": RESULTS_06_DIR / "notebook06_image_level_anatomical_proxy_label_statistics.csv",
    "patient_level_statistics": RESULTS_06_DIR / "notebook06_patient_level_anatomical_proxy_label_statistics.csv",
    "statistical_summary": RESULTS_06_DIR / "notebook06_anatomical_proxy_statistical_summary.json",
    "figure_manifest": RESULTS_06_DIR / "notebook06_anatomical_proxy_figure_manifest.csv",
    "main_figure_v2_manifest": RESULTS_06_DIR / "notebook06_main_anatomical_proxy_figure_v2_manifest.csv",
    "main_figure_v2_summary": RESULTS_06_DIR / "notebook06_main_anatomical_proxy_figure_v2_summary.json",
}

file_checks = {
    key: path.exists()
    for key, path in required_files.items()
}

missing_files = [
    key for key, exists in file_checks.items()
    if not exists
]

if missing_files:
    raise FileNotFoundError(f"Missing required Notebook 06 artifacts: {missing_files}")

proxy_qc = pd.read_csv(required_files["final_proxy_manifest_with_qc"])
analysis_manifest = pd.read_csv(required_files["analysis_manifest"])
patient_proxy_summary = pd.read_csv(required_files["patient_proxy_summary"])
failed_proxy_cases = pd.read_csv(required_files["failed_proxy_cases"])
low_confidence_cases = pd.read_csv(required_files["low_confidence_cases"])
patient_level_statistics = pd.read_csv(required_files["patient_level_statistics"])
image_level_statistics = pd.read_csv(required_files["image_level_statistics"])
main_v2_manifest = pd.read_csv(required_files["main_figure_v2_manifest"])

with open(required_files["statistical_summary"], "r", encoding="utf-8") as f:
    statistical_summary = json.load(f)

with open(required_files["main_figure_v2_summary"], "r", encoding="utf-8") as f:
    main_figure_v2_summary = json.load(f)

required_proxy_columns = [
    "strict_anatomical_proxy_valid",
    "low_confidence_anatomical_proxy",
]

missing_proxy_columns = [
    col for col in required_proxy_columns
    if col not in proxy_qc.columns
]

if missing_proxy_columns:
    raise ValueError(f"Missing required proxy QC columns: {missing_proxy_columns}")

proxy_qc["strict_anatomical_proxy_valid"] = as_bool_series(
    proxy_qc["strict_anatomical_proxy_valid"]
)

proxy_qc["low_confidence_anatomical_proxy"] = as_bool_series(
    proxy_qc["low_confidence_anatomical_proxy"]
)

strict_valid_mask = proxy_qc["strict_anatomical_proxy_valid"]
low_confidence_mask = proxy_qc["low_confidence_anatomical_proxy"]
high_confidence_mask = strict_valid_mask & (~low_confidence_mask)

strict_valid_count = int(strict_valid_mask.sum())
high_confidence_count = int(high_confidence_mask.sum())
low_confidence_count = int(low_confidence_mask.sum())
failed_count = int((~strict_valid_mask).sum())

main_v2_manifest["exists"] = as_bool_series(main_v2_manifest["exists"])
main_v2_files_exist = bool(main_v2_manifest["exists"].all())

primary_patient_row = patient_level_statistics[
    patient_level_statistics["metric"] == "area_cdr_proxy_mean_strict"
]

if len(primary_patient_row) != 1:
    raise RuntimeError("Primary patient-level statistic not found.")

primary_patient_row = primary_patient_row.iloc[0]

content_checks = {
    "expected_proxy_rows": len(proxy_qc) == 747,
    "expected_analysis_rows": len(analysis_manifest) == 747,
    "expected_patients": len(patient_proxy_summary) == 288,
    "expected_strict_valid_count": strict_valid_count == 745,
    "expected_high_confidence_count": high_confidence_count == 716,
    "expected_low_confidence_count": low_confidence_count == 29,
    "expected_failed_count": failed_count == 2,
    "expected_failed_case_rows": len(failed_proxy_cases) == 2,
    "expected_low_confidence_case_rows": len(low_confidence_cases) == 29,
    "main_figure_v2_files_exist": main_v2_files_exist,
    "patient_level_gon_positive_higher_than_negative": primary_patient_row["mean_difference_positive_minus_negative"] > 0,
    "patient_level_statistically_significant": primary_patient_row["mann_whitney_p"] < 0.05,
}

final_checks = {**file_checks, **content_checks}

failed_checks = [
    key for key, value in final_checks.items()
    if not bool(value)
]

status = "PASS" if len(failed_checks) == 0 else "FAIL"

notebook06_status = {
    "notebook": "06_apply_final_od_oc_segmentor_to_hygd_anatomical_proxy",
    "status": status,
    "final_segmentor_model": FINAL_MODEL_NAME,
    "final_checkpoint": str(FINAL_CHECKPOINT_PATH),
    "total_hygd_images": int(len(proxy_qc)),
    "total_patients": int(len(patient_proxy_summary)),
    "strict_valid_anatomical_proxies": strict_valid_count,
    "high_confidence_anatomical_proxies": high_confidence_count,
    "low_confidence_valid_proxies": low_confidence_count,
    "failed_anatomical_proxies": failed_count,
    "primary_patient_level_metric": "area_cdr_proxy_mean_strict",
    "primary_patient_level_gon_positive_mean": float(primary_patient_row["positive_mean"]),
    "primary_patient_level_gon_negative_mean": float(primary_patient_row["negative_mean"]),
    "primary_patient_level_difference": float(primary_patient_row["mean_difference_positive_minus_negative"]),
    "primary_patient_level_ci95": [
        float(primary_patient_row["mean_difference_ci95_low"]),
        float(primary_patient_row["mean_difference_ci95_high"]),
    ],
    "primary_patient_level_mann_whitney_p": float(primary_patient_row["mann_whitney_p"]),
    "primary_patient_level_rank_biserial": float(primary_patient_row["rank_biserial_effect_size"]),
    "main_figure_v2_png": main_figure_v2_summary["main_figure_v2_png"],
    "main_figure_v2_pdf": main_figure_v2_summary["main_figure_v2_pdf"],
    "main_figure_v2_svg": main_figure_v2_summary["main_figure_v2_svg"],
    "analysis_manifest": str(required_files["analysis_manifest"]),
    "patient_proxy_summary": str(required_files["patient_proxy_summary"]),
    "failed_checks": failed_checks,
    "next_notebook": "07_train_or_load_hygd_glaucoma_classifiers_patient_safe.ipynb",
}

with open(FINAL_STATUS_JSON, "w", encoding="utf-8") as f:
    json.dump(notebook06_status, f, indent=2)

with open(FINAL_STATUS_TXT, "w", encoding="utf-8") as f:
    for key, value in notebook06_status.items():
        f.write(f"{key}: {value}\n")

pd.DataFrame(
    [{"check": key, "passed": bool(value)} for key, value in final_checks.items()]
).to_csv(FINAL_STATUS_CHECKS_CSV, index=False)

print(f"Status: {status}")
print(f"Final segmentor: {FINAL_MODEL_NAME}")
print(f"HYGD images: {len(proxy_qc)}")
print(f"Patients: {len(patient_proxy_summary)}")
print(f"Strict valid proxies: {strict_valid_count}")
print(f"High-confidence proxies: {high_confidence_count}")
print(f"Low-confidence valid proxies: {low_confidence_count}")
print(f"Failed proxies: {failed_count}")
print(
    "Patient-level CDR proxy difference GON+ - GON−: "
    f"{primary_patient_row['mean_difference_positive_minus_negative']:.6f}"
)
print(
    "95% CI: "
    f"[{primary_patient_row['mean_difference_ci95_low']:.6f}, "
    f"{primary_patient_row['mean_difference_ci95_high']:.6f}]"
)
print(f"p-value: {primary_patient_row['mann_whitney_p']:.3e}")
print(f"Main figure v2: {main_figure_v2_summary['main_figure_v2_png']}")
print(f"Final status: {FINAL_STATUS_JSON}")

if failed_checks:
    raise RuntimeError(f"Notebook 06 failed checks: {failed_checks}")